# Embedding models comparison pipeline

In [1]:
import os
from pathlib import Path

from abc import ABC, abstractmethod

import torch
from torch import nn
from torch.utils.data import Dataset, DataLoader
import torch.nn.functional as F
from torch.optim.lr_scheduler import ReduceLROnPlateau

import transformers
from tokenizers import BertWordPieceTokenizer
from transformers import BertConfig, BertTokenizer, AutoTokenizer, AutoModel, get_linear_schedule_with_warmup

import sklearn.metrics as skmetrics
from sklearn.metrics import roc_auc_score, average_precision_score

from ray import train, tune
from ray.tune import ResultGrid

from typing import List, Dict, Union, Any, Optional

import re
import random
import itertools
import math
import time
from datetime import datetime
import numpy as np
import pandas as pd
from tqdm.notebook import trange, tqdm
from sklearn.model_selection import train_test_split
from imblearn.over_sampling import RandomOverSampler

In [2]:
device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
device

device(type='cuda', index=0)

In [3]:
torch.cuda.empty_cache()

## Modules

### Utils modules

In [16]:
import sklearn.metrics as skmetrics
from sklearn.metrics import confusion_matrix, accuracy_score

def check_input_type(labels: Union[np.ndarray, List[float]], 
                     preds_scores: Union[np.ndarray, List[float]], 
                     threshold: float) -> (np.ndarray, np.ndarray):
    """
    Check and convert input types to numpy arrays and validate their shapes.

    :param labels: Ground truth labels.
    :type labels: Union[np.ndarray, List[float]]
    :param preds_scores: Predicted scores.
    :type preds_scores: Union[np.ndarray, List[float]]
    :param threshold: Threshold for converting scores to binary labels.
    :type threshold: float
    :return: Ground truth labels and predicted scores as numpy arrays.
    :rtype: (np.ndarray, np.ndarray)
    :raises ValueError: If predictions and labels are not in the same shape or not 1D arrays.
    """
    if not isinstance(preds_scores, np.ndarray):
        preds_scores = np.array(preds_scores)
    if not isinstance(labels, np.ndarray):
        labels = np.array(labels)
    if preds_scores.shape[0] != labels.shape[0]:
        raise ValueError("Predictions and labels are not in the same shape")
    if preds_scores.ndim != 1 or labels.ndim != 1:
        labels = labels.reshape(-1)
        preds_scores = preds_scores.reshape(-1)
    
    return labels, preds_scores

def evaluate_scores(labels: Union[np.ndarray, List[float]], 
                    preds_scores: Union[np.ndarray, List[float]], 
                    eval_metrics: Union[str, List[str]], 
                    threshold: float = 0.5) -> Dict[str, Any]:
    """
    Evaluate prediction scores using specified metrics.

    :param labels: Ground truth labels.
    :type labels: Union[np.ndarray, List[float]]
    :param preds_scores: Predicted scores.
    :type preds_scores: Union[np.ndarray, List[float]]
    :param eval_metrics: Evaluation metrics to be calculated.
    :type eval_metrics: Union[str, List[str]]
    :param threshold: Threshold for converting scores to binary labels, defaults to 0.5.
    :type threshold: float, optional
    :return: Dictionary containing evaluation results.
    :rtype: Dict[str, Any]
    :raises ValueError: If an unsupported metric is provided.
    """
    
    if isinstance(eval_metrics, str): 
        eval_metrics = [eval_metrics]
        
    labels, preds_scores = check_input_type(labels, preds_scores, threshold = threshold)
    preds_labels = apply_threshold(preds_scores, threshold)
    
    results = {}
    unique_classes = np.unique(labels)
    tn, fp, fn, tp = confusion_matrix(labels, preds_labels).ravel()
    for metric in eval_metrics:
        
        if metric == 'confusion_matrix':
            results['tn'], results['fp'], results['fn'], results['tp'] = tn, fp, fn, tp
        elif metric == 'specificity':
            results['specificity'] = tn / (tn+fp)
        elif metric == 'npv':
            results['npv'] = tn / (tn+fn)
        elif metric == 'fnr':
            results['fnr'] = fn / (tp+fn)
        elif metric == 'lift':
            results['lift'] = (tp/(tp+fp))/((tp+fn)/(tp+tn+fp+fn))
        elif metric == 'lift@1percent':
            results['lift@1percent'] = lift_at_1_percent(labels, preds_scores)
        elif metric == 'precision@1percent':
            results['precision@1percent'] = precision_at_1_percent(labels, preds_scores)
        elif metric == 'recall@1percent':
            results['recall@1percent'] = true_positive_at_1_percent(labels, preds_scores)
        elif 'auc' in metric: # roc_auc, pr_auc
            if len(unique_classes) == 1:
                results[metric] = float('nan')  # or return a default value
                continue
            metric_func = getattr(skmetrics, metric)
            results[metric] = metric_func(labels, preds_scores)
        elif hasattr(skmetrics, metric):
            metric_func = getattr(skmetrics, metric)
            results[metric] = metric_func(labels, preds_labels)
        else:
            raise ValueError(f"Unsupported metric: {metric}")

    return results

def evaluate_scores_epochs(epochs_labels: List[Union[np.ndarray, List[float]]], 
                           epochs_preds_scores: List[Union[np.ndarray, List[float]]], 
                           eval_metrics: Union[str, List[str]],
                           threshold: float = 0.5) -> Dict[str, List[Any]]:
    """
    Evaluate prediction scores for multiple epochs using specified metrics.

    :param epochs_labels: List of ground truth labels for each epoch.
    :type epochs_labels: List[Union[np.ndarray, List[float]]]
    :param epochs_preds_scores: List of predicted scores for each epoch.
    :type epochs_preds_scores: List[Union[np.ndarray, List[float]]]
    :param eval_metrics: Evaluation metrics to be calculated.
    :type eval_metrics: Union[str, List[str]]
    :param threshold: Threshold for converting scores to binary labels, defaults to 0.5.
    :type threshold: float, optional
    :return: Dictionary containing evaluation results for each epoch.
    :rtype: Dict[str, List[Any]]
    :raises ValueError: If the number of epochs in labels and predictions do not match.
    """
    
    if len(epochs_labels) != len(epochs_preds_scores):
        raise ValueError(f"{len(epochs_labelss)} != {len(epochs_preds_scores)}")
        
    metrics_scores = {} 
    for i in range(len(epochs_labels)):
        res = evaluate_scores(epochs_labels[i], 
                             epochs_preds_scores[i], 
                             eval_metrics, 
                             threshold = threshold)
        for k, v in res.items():
            metrics_scores.setdefault(k, []).append(v)
    return metrics_scores

def apply_threshold(values: Union[np.ndarray, List[float]], 
                    threshold: float) -> List[int]:
    """
    Apply a threshold to a list of values to convert them to binary labels.

    :param values: List of values to be thresholded.
    :type values: Union[np.ndarray, List[float]]
    :param threshold: Threshold for converting values to binary labels.
    :type threshold: float
    :return: List of binary labels.
    :rtype: List[int]
    :raises ValueError: If the threshold is not between 0 and 1.
    """
    
    if not 0 < threshold < 1:
        raise ValueError("Percentage must be between 0 and 100")

    # Calculate the number of values to set to 1
    num_positives = int(len(values) * threshold)

    if num_positives == 0:
        return [0] * len(values)
        
    # Find the cutoff value
    sorted_values = sorted(values, reverse=True)
    cutoff_value = sorted_values[num_positives - 1]

    # Create the binary list based on the cutoff value
    binary_list = [1 if value >= cutoff_value else 0 for value in values]

    return binary_list

def lift_at_x_percent(y_true: np.ndarray, y_scores: np.ndarray, ratio = 0.01):
    top_1_percent = max(1, int(ratio * len(y_true)))
    top_indices = np.argsort(-y_scores)[:top_1_percent]  # Use -y_scores to get the indices in descending order
    actual_positives = np.sum(y_true[top_indices])
    expected_positives = np.sum(y_true) * ratio
    lift = actual_positives / expected_positives if expected_positives != 0 else 0
    return lift

def true_positive_at_x_percent(y_true: np.ndarray, y_scores: np.ndarray, ratio = 0.01):
    top_1_percent = max(1, int(ratio * len(y_true)))
    top_indices = np.argsort(-y_scores)[:top_1_percent]
    recall = np.sum(y_true[top_indices])/np.sum(y_true)
    return recall

def precision_at_x_percent(y_true: np.ndarray, y_scores: np.ndarray, ratio = 0.01):
    top_1_percent = max(1, int(ratio * len(y_true)))
    top_indices = np.argsort(-y_scores)[:top_1_percent]
    ppv = np.sum(y_true[top_indices])/top_1_percent
    return ppv

def num_samples_at_x_percent(y_true: np.ndarray, y_scores: np.ndarray, ratio = 0.01):
    top_1_percent = max(1, int(ratio * len(y_true)))
    return top_1_percent

def f1_score_top_x_percent(y_true: np.ndarray, y_pred_proba: np.ndarray, ratio = 0.01):
    from sklearn.metrics import f1_score
    top_1_percent = int(ratio * len(y_true))
    top_indices = np.argsort(-y_pred_proba)[:top_1_percent]
    y_true_top = y_true[top_indices]
    y_pred_top = np.ones(top_1_percent)
    return f1_score(y_true_top, (y_pred_top > 0).astype(int))  # Use threshold to convert probabilities to labels

In [16]:
f1_score_top_1_percent(np.array([[1], [0], [0], [1], [0], [1]]).reshape(-1), np.array([0.8, 0.1, 0.2, 0.9, 0.1, 0.8]))

/opt/conda/envs/pytorch/lib/python3.10/site-packages/sklearn/metrics/_classification.py:1517: UndefinedMetricWarning: F-score is ill-defined and being set to 0.0 due to no true nor predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


0.0

### Emedding model modules

* `SentenceTransformerEmbedding`
  1. Consider enabling more parameters
  2. Consider allowing finetuning sentenceTransformer object
 
* `GeckoEmbedding`:
  1. Further implement authentication function to allow user to specify access
  2. Reimplement the encode function to allow greater batch size

* `BERTEmbedding`:
  1. Pretrained BERT model

In [5]:
import os
import time
import numpy as np
import torch
from datetime import datetime
from abc import ABC, abstractmethod
from torch.utils.data import DataLoader
from sentence_transformers import SentenceTransformer
from transformers import AutoTokenizer, AutoModel
import vertexai
from vertexai.preview.language_models import TextEmbeddingModel
from typing import List, Union, Optional

class Text2Embedding:

    @abstractmethod
    def encode(self, texts: List[str]) -> np.ndarray:
        """
        Encode a list of texts into embeddings.

        :param texts: List of texts to encode.
        :type texts: List[str]
        :return: Encoded embeddings.
        :rtype: np.ndarray
        """
        pass

    def save_embed(self, 
                   embedding: np.ndarray, 
                   save_path: Optional[str] = None, 
                   file_name: Optional[str] = None) -> None:
        """
        Save the embedding to a file.

        :param embedding: Embedding to save.
        :type embedding: np.ndarray
        :param save_path: Path to save the embedding, defaults to None.
        :type save_path: Optional[str], optional
        :param file_name: Name of the file to save the embedding, defaults to None.
        :type file_name: Optional[str], optional
        """
        
        # check and ensure the specified path exists
        # if it's none then set up default save path
        if save_path is None: 
            save_path = os.path.join(os.getcwd(), 'embedding_outputs')

        # if specified path does not exist, then save to the specified path
        if not os.path.exists(save_path):
            os.makedirs(save_path)

        if file_name[-4:] != '.npy':
            file_name = filename + '.npy '
        
        full_path = os.path.join(save_path, file_name)
        np.save(full_path, embedding)
        print(f"Embedding saved to {full_path}")

class SentenceTransformerEmbedding(Text2Embedding):
    """
    Encode text to embedding using sentence transformer
    """
    def __init__(self, 
                 model_name: str, 
                 multiprocessing: bool = False,
                 **kwargs) -> None:
        """
        Initialize the SentenceTransformerEmbedding class.

        :param model_name: Name of the Sentence Transformer model.
        :type model_name: str
        :param multiprocessing: Whether to use multiprocessing, defaults to False.
        :type multiprocessing: bool, optional
        """
        self.model_name = model_name
        self.multiprocessing = multiprocessing
        self.model = SentenceTransformer(model_name, **kwargs)

    def encode(self, texts: List[str]) -> np.ndarray:
        """
        Encode a list of texts into embeddings.

        :param texts: List of texts to encode.
        :type texts: List[str]
        :return: Encoded embeddings.
        :rtype: np.ndarray
        """
        if self.multiprocessing:
            pool = self.model.start_multi_process_pool()
            embed_output = self.model.encode_multi_process(texts, pool)
            model.stop_multi_process_pool(pool)
        else:
            embed_output = self.model.encode(texts, show_progress_bar = True)
        self.embed_output = embed_output
        return embed_output

    def save_embed(self, save_path: Optional[str] = None, file_name: Optional[str] = None) -> None:
        """
        Save the embedding to a file.

        :param save_path: Path to save the embedding, defaults to None.
        :type save_path: Optional[str], optional
        :param file_name: Name of the file to save the embedding, defaults to None.
        :type file_name: Optional[str], optional
        """
        if not file_name: 
            timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
            file_name = f"ST_embedding_{self.model_name}_{timestamp}.npy"
        super().save_embed(self.embed_output, save_path = save_path, file_name = file_name)

class GeckoEmbedding(Text2Embedding):
    """

    project_name = "anbc-dev-hcm-cm-de"
    location = "us-east4"
    model_name = "text-embedding-004"
    """
    def __init__(self,
                 model_name: str,
                 project_name: str,
                 location: str) -> None:
        """
        Initialize the GeckoEmbedding class.

        :param model_name: Name of the Gecko model.
        :type model_name: str
        :param project_name: Name of the project.
        :type project_name: str
        :param location: Location of the project.
        :type location: str
        """
        self.model_name = model_name
        self._authentication(project_name=project_name, location=location)
        
    def _authentication(self, project_name: str, location: str) -> None:
        """
        Authenticate with the Vertex AI service.

        :param project_name: Name of the project.
        :type project_name: str
        :param location: Location of the project.
        :type location: str
        """
        try: 
            vertexai.init(project=project_name, 
                          location=location)
        except Exception as e:
            print(e)

    def encode(self, 
               texts: List[str], 
               embed_dimension: Optional[int] = None, 
               batch_size: int = 10) -> np.ndarray:
        """
        Encode a list of texts into embeddings.

        :param texts: List of texts to encode.
        :type texts: List[str]
        :param embed_dimension: Dimension of the embeddings, defaults to None.
        :type embed_dimension: Optional[int], optional
        :param batch_size: Batch size for encoding, defaults to 10.
        :type batch_size: int, optional
        :return: Encoded embeddings.
        :rtype: np.ndarray
        """

        embed_output = []
        start_time = time.time()
        embed_model = TextEmbeddingModel.from_pretrained(self.model_name)
        kwargs = dict(output_dimensionality=embed_dimension) if embed_dimension else {}

        embed_output = []
        for i in tqdm(range(0, len(texts), batch_size)):
            batch = texts[i: i+batch_size]
            embed_batch = embed_model.get_embeddings(batch, **kwargs)
            embed_output.extend(embed_batch)
            
            if i > 0 and i % 300 == 0 and time.time() - start_time < 60:
                time.sleep(60 - (time.time() - start_time))
                start_time = time.time()
        embed_output = np.asarray([embed_output[i].values for i in range(len(embed_output))])
        self.embed_output = embed_output
        return embed_output

    def save_embed(self, save_path: Optional[str] = None, file_name: Optional[str] = None) -> None:
        """
        Save the embedding to a file.

        :param save_path: Path to save the embedding, defaults to None.
        :type save_path: Optional[str], optional
        :param file_name: Name of the file to save the embedding, defaults to None.
        :type file_name: Optional[str], optional
        """
        if not file_name:
            timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
            file_name = f"Gecko_embedding_{self.model_name}_{timestamp}.npy"
        super().save_embed(self.embed_output, save_path = save_path, file_name = file_name)



class BERTEmbedding(Text2Embedding):
    
    def __init__(self,
                 model_name: str,
                 max_length: int,
                 embedding_type: str = 'cls_token') -> None:
        """
        Initialize the BERTEmbedding class.

        :param model_name: Name of the BERT model.
        :type model_name: str
        :param max_length: Maximum length of the tokenized input.
        :type max_length: int
        :param embedding_type: Type of embedding to extract, defaults to 'cls_token'.
        :type embedding_type: str, optional
        """
        self.model_name = model_name
        self.max_length = max_length
        self.embedding_type = embedding_type
        self.tokenizer = AutoTokenizer.from_pretrained(self.model_name)
        self.device = torch.device("cuda") if torch.cuda.is_available() else torch.device("cpu")
        self.model = AutoModel.from_pretrained(self.model_name).to(self.device)

    # here add finetuning and get embedding 

    def encode(self, texts: List[str], batch_size: int = 16) -> np.ndarray:
        """
        Encode a list of texts into embeddings.

        :param texts: List of texts to encode.
        :type texts: List[str]
        :param batch_size: Batch size for encoding, defaults to 16; greater value is likely to lead to Out of Memory error. 
        :type batch_size: int, optional
        :return: Encoded embeddings.
        :rtype: np.ndarray
        """
        
        # Convert the list of texts into a DataLoader
        data_loader = DataLoader(texts, batch_size=batch_size)
    
        embed_output = []
        for batch in tqdm(data_loader):
            # Tokenize the batch of texts
            encodings = self.tokenizer.batch_encode_plus(batch, 
                                                        return_tensors='pt', 
                                                        max_length=self.max_length, 
                                                        padding='max_length', 
                                                        truncation=True)
            input_ids = encodings['input_ids'].to(self.device)
            attention_mask = encodings['attention_mask'].to(self.device)
            outputs = self.model(input_ids=input_ids, 
                                 attention_mask=attention_mask)
            if self.embedding_type == 'pooler_output': # a linear projection of cls emdifferent than cls embedding
                embeddings = outputs.pooler_output
            elif self.embedding_type == 'cls_token': # cls embedding
                embeddings = outputs.last_hidden_state[:, 0, :]
            elif self.embedding_type == 'last_hidden_state': # avg. token embedding
                # Calculate the average only over non-padding tokens
                mask_expanded = attention_mask.unsqueeze(-1).expand(outputs.last_hidden_state.size()).float()
                sum_embeddings = torch.sum(outputs.last_hidden_state * mask_expanded, 1)
                sum_mask = mask_expanded.sum(1) # This counts the number of non-padding tokens
                sum_mask = torch.clamp(sum_mask, min=1e-9) # Prevent division by zero
                embeddings = sum_embeddings / sum_mask
            else:
                raise NameError(f"{self.embedding_type} does not exist")
                
            if isinstance(embeddings, torch.Tensor):
                embeddings = np.asarray(embeddings.cpu().detach())
                embed_output.append(embeddings)
                
            del encodings, embeddings, input_ids, attention_mask, outputs
            torch.cuda.empty_cache()
            
        embed_output = np.vstack(embed_output)
        self.embed_output = embed_output
        return embed_output

    def save_embed(self, save_path: Optional[str] = None, file_name: Optional[str] = None) -> None:
        """
        Save the embedding to a file.

        :param save_path: Path to save the embedding, defaults to None.
        :type save_path: Optional[str], optional
        :param file_name: Name of the file to save the embedding, defaults to None.
        :type file_name: Optional[str], optional
        """

        if not file_name:
            timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
            file_name = f"BERT_embedding_{self.model_name}_{timestamp}.npy"
        super().save_embed(self.embed_output, save_path = save_path, file_name = file_name)

2024-08-05 13:31:19.048422: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:485] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
2024-08-05 13:31:19.070267: E external/local_xla/xla/stream_executor/cuda/cuda_dnn.cc:8454] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
2024-08-05 13:31:19.076865: E external/local_xla/xla/stream_executor/cuda/cuda_blas.cc:1452] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
2024-08-05 13:31:19.094487: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.


RuntimeError: Failed to import transformers.integrations.integration_utils because of the following error (look up to see its traceback):
Failed to import transformers.modeling_tf_utils because of the following error (look up to see its traceback):
Your currently installed version of Keras is Keras 3, but this is not yet supported in Transformers. Please install the backwards-compatible tf-keras package with `pip install tf-keras`.

In [ ]:
from typing import Sequence, Dict, Any

def encode_texts(texts: Sequence[str],
                 model: Text2Embedding,
                 **kwargs) -> np.ndarray:
    """
    Encode a sequence of texts using a specified model and optionally save the embeddings.

    :param texts: Sequence of texts to encode.
    :type texts: Sequence[str]
    :param model: Model to use for encoding.
    :type model: Text2Embedding
    :param kwargs: Additional keyword arguments for encoding and saving.
    :return: Encoded embeddings.
    :rtype: np.ndarray
    """
    
    embed_outputs = model.encode(texts, **kwargs)
    if 'save_path' in kwargs and 'file_name' in kwargs:
        model.save_embed(save_path = kwargs['save_path'], filename = kwargs['file_name'])
        
    return embed_outputs

#### Test case

In [55]:
sentences = [
    "Three years later, the coffin was still full of Jello.",
    "The fish dreamed of escaping the fishbowl and into the toilet where he saw his friend go.",
    "The person box was packed with jelly many dozens of months later.",
    "He found a leprechaun in his walnut shell."
]

In [56]:
# test gecko model
model_name = "text-embedding-004"
project_name = "anbc-dev-hcm-cm-de"
location = "us-east4"
gecko_embedding = GeckoEmbedding(model_name, 
                                 project_name = project_name, 
                                 location = location)
gecko_embedding.encode(sentences)

  0%|          | 0/1 [00:00<?, ?it/s]

array([[ 0.04238003, -0.00406774,  0.04524068, ..., -0.04238391,
         0.0050767 , -0.06078254],
       [ 0.00621137,  0.07721061,  0.00979152, ...,  0.01041247,
         0.01910078, -0.07883814],
       [ 0.05433393,  0.05857591,  0.04541778, ..., -0.03686114,
         0.00708915, -0.05669801],
       [-0.04048022, -0.00759808, -0.00678168, ..., -0.0342277 ,
        -0.04585627, -0.03781197]])

array([[ 0.04238003, -0.00406774,  0.04524068, ..., -0.04238391,
         0.0050767 , -0.06078254],
       [ 0.00621137,  0.07721061,  0.00979152, ...,  0.01041247,
         0.01910078, -0.07883814],
       [ 0.05433393,  0.05857591,  0.04541778, ..., -0.03686114,
         0.00708915, -0.05669801],
       [-0.04048022, -0.00759808, -0.00678168, ..., -0.0342277 ,
        -0.04585627, -0.03781197]])

In [9]:
# test BERT model
model_name = 'bert-base-uncased'
max_length = 512
embedding_type = 'cls_token' # or 'pooler_output'
bert_embedding = BERTEmbedding(model_name, max_length, embedding_type)

embeddings = bert_embedding.encode(sentences)
print(embeddings)

/home/a964286/.local/lib/python3.9/site-packages/huggingface_hub/file_download.py:1132: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(


  0%|          | 0/1 [00:00<?, ?it/s]

[[-0.04392279 -0.13256723  0.0305367  ... -0.15951952  0.57552826
   0.38231143]
 [-0.05837959 -0.1955893  -0.10775114 ... -0.31821465  0.26602656
   0.5726912 ]
 [-0.27212337 -0.13840866  0.06878543 ... -0.2759227   0.4904628
   0.33208448]
 [-0.17499708  0.11010065 -0.16292106 ... -0.2086643   0.3560317
   0.29989073]]


In [ ]:
# test SentenceTransformer
ste = SentenceTransformerEmbedding(model_name = 'all-MiniLM-L6-v2',
                                  multiprocessing = False)
res = ste.encode(sentences)

### Embedding model registry module

In [5]:
VALID_MODEL_NAMES = {
                    'hf_bert': ['bert-base-uncased', 'emilyalsentzer/Bio_ClinicalBERT', 'yikuan8/Clinical-Longformer', 'medicalai/ClinicalBERT'],
                    'sentence_transformer': ['all-MiniLM-L6-v2', 'all-mpnet-base-v2', 'infgrad/stella_en_400M_v5', 'infgrad/stella_en_1.5B_v5'],
                    'gecko_embedding': ['text-embedding-004', 'textembedding-gecko@003']
                    }

In [6]:
class SentenceTransformerEmbeddingModelLoader:
    @staticmethod
    def valid_model_name(model_name):
        if model_name in VALID_MODEL_NAMES['sentence_transformer']:
            return True
        return False

    @staticmethod
    def load_model(model_name, multiprocessing=False, **kwargs):
        return SentenceTransformerEmbedding(model_name, multiprocessing, **kwargs)

class GeckoEmbeddingModelLoader:
    @staticmethod
    def valid_model_name(model_name):
        if model_name in VALID_MODEL_NAMES['gecko_embedding']:
            return True
        return False

    @staticmethod
    def load_model(model_name, project_name, location):
        return GeckoEmbedding(model_name, project_name, location)

class BERTEmbeddingModelLoader:
    @staticmethod
    def valid_model_name(model_name):
        if model_name in VALID_MODEL_NAMES['hf_bert']:
            return True
        return False

    @staticmethod
    def load_model(model_name, 
                   max_length, 
                   embedding_type='cls_token'): # 'pooler_output'
        return BERTEmbedding(model_name, 
                             max_length = max_length, 
                             embedding_type = embedding_type)

class EmbeddingModelRegistry:
    def __init__(self):
        self.loaders = [SentenceTransformerEmbeddingModelLoader, 
                        GeckoEmbeddingModelLoader, 
                        BERTEmbeddingModelLoader]

    def load_model(self, model_name, **kwargs):
        for loader in self.loaders:
            if loader.valid_model_name(model_name):
                try:
                    return loader.load_model(model_name, **kwargs)
                except Exception as e:
                    print(e)
                    continue
        raise NameError(f"The model '{model_name}' does not exist in the known model hubs.")

In [87]:
test_model_name = 'infgrad/stella_en_400M_v5'
embed_model_registry = EmbeddingModelRegistry()
test_model = embed_model_registry.load_model(model_name = test_model_name, trust_remote_code = True)

configuration.py:   0%|          | 0.00/7.13k [00:00<?, ?B/s]

A new version of the following files was downloaded from https://huggingface.co/infgrad/stella_en_400M_v5:
- configuration.py
. Make sure to double-check they do not contain any added malicious code. To avoid downloading new versions of the code file, you can pin a revision.


modeling.py:   0%|          | 0.00/57.5k [00:00<?, ?B/s]

A new version of the following files was downloaded from https://huggingface.co/infgrad/stella_en_400M_v5:
- modeling.py
. Make sure to double-check they do not contain any added malicious code. To avoid downloading new versions of the code file, you can pin a revision.


pytorch_model.bin:   0%|          | 0.00/1.74G [00:00<?, ?B/s]

please install xformers


NameError: The model 'infgrad/stella_en_400M_v5' does not exist in the known model hubs.

In [51]:
test_model

### Dataset module

In [17]:
class EmbeddingDatasets(torch.utils.data.Dataset):

    def __init__(self,
                 embeddings: np.ndarray,
                 labels: np.ndarray = None) -> None:
        """
        Initialize the EmbeddingDatasets class.

        :param embeddings: The embeddings dataset.
        :type embeddings: np.ndarray
        :param labels: The labels dataset.
        :type labels: np.ndarray
        """
        super().__init__()
        self.embeddings = embeddings
        self.labels = labels
        self.input_size = embeddings.shape[1]
    def __len__(self):
        return len(self.embeddings)

    def __getitem__(self, idx: int) -> Dict[str, torch.Tensor]:
        """
        Get a sample from the dataset.

        :param idx: The index of the sample.
        :type idx: int
        :return: A dictionary containing the embedding and label.
        :rtype: Dict[str, torch.Tensor]
        """
        output = {
            'embedding': torch.tensor(self.embeddings[idx])
        }
        
        if self.labels is not None:
            output['label'] = torch.tensor(self.labels[idx])

        return output

### Models

#### DNN model module

In [12]:
class DNNClassifier(nn.Module):
    """
    A generic classification that intake embeddings only (Sentence-transformer, LLM-based model)
    
    """
    def __init__(self, 
                 input_size: Optional[int] = None,
                 hidden_sizes: List[int] = [],
                 dropout_rate: float = 0.2) -> None:
        """
        Initialize the DNNClassifier class.

        :param input_size: Size of the input embeddings, defaults to None
        :type input_size: Optional[int], optional
        :param hidden_sizes: List of hidden layer sizes
        :type hidden_sizes: List[int], optional
        :param dropout_rate: Dropout rate for the dropout layer, defaults to 0.2
        :type dropout_rate: float, optional
        """
        
        super().__init__()
        self.input_size = input_size
        self.hidden_sizes = hidden_sizes
        self.dropout_rate = dropout_rate
        self._build_model()


    def _build_model(self):
        """Builds the model with the current parameters."""
        self._linear_act_layers = nn.ModuleList()
        self._layer_sizes = [self.input_size] + list(self.hidden_sizes) + [1]
        for i in range(len(self._layer_sizes)-1):
            self._linear_act_layers.append(nn.Linear(self._layer_sizes[i], self._layer_sizes[i+1]))
            self._linear_act_layers.append(nn.BatchNorm1d(self._layer_sizes[i+1]))
            if i < len(self._layer_sizes)-2:
                self._linear_act_layers.append(nn.ReLU())
                self._linear_act_layers.append(nn.Dropout(self.dropout_rate))
        self._sigmoid = nn.Sigmoid()
        
    def set_architecture(self,
                         hidden_sizes: List[int],
                         dropout_rate: float) -> None:
        """
        Sets the architecture of the model and rebuilds it.

        :param hidden_sizes: List of hidden layer sizes
        :type hidden_sizes: List[int]
        :param dropout_rate: Dropout rate for the dropout layer
        :type dropout_rate: float
        """
        self.hidden_sizes = hidden_sizes
        self.dropout_rate = dropout_rate
        self._build_model()
    
    def forward(self, x: torch.Tensor) -> torch.Tensor:
        """
        Performs a forward pass through the model.

        :param x: Input tensor
        :type x: torch.Tensor
        :return: Output tensor after passing through the model
        :rtype: torch.Tensor
        """
        for layer in self._linear_act_layers:
            x = layer(x)
        output = self._sigmoid(x)
        return output

#### DNN model test module

In [33]:
import unittest
import torch
from torch import nn
from typing import List

class TestDNNClassifier(unittest.TestCase):
    def setUp(self):
        self.model = DNNClassifier(input_size=10, hidden_sizes=[20, 10], dropout_rate=0.1)

    def test_forward(self):
        input_tensor = torch.randn(1, 10)
        output = self.model(input_tensor)
        self.assertEqual(output.size(), (1, 1))

    def test_set_architecture(self):
        self.model.set_architecture(hidden_sizes=[30, 15], dropout_rate=0.2)
        input_tensor = torch.randn(1, 10)
        output = self.model(input_tensor)
        self.assertEqual(output.size(), (1, 1))

    def test_model_rebuild(self):
        self.model.set_architecture(hidden_sizes=[30, 15], dropout_rate=0.2)
        self.assertEqual(len(self.model._linear_act_layers), 7)  # 3 Linear, 2 Activation, 2 Dropout

unittest.main(argv=['first-arg-is-ignored'], exit=False)

...
----------------------------------------------------------------------
Ran 3 tests in 0.010s

OK


### Trainer

#### Model trainer

In [5]:
import torch
import time
from tqdm import tqdm
import sklearn.metrics as skmetrics
import numpy as np
from typing import List, Tuple, Any

class ModelTrainer:
    """
    A class to train and validate a machine learning model
    """
    def __init__(self, 
                 ml_model: torch.nn.Module, 
                 optimizer: torch.optim.Optimizer, 
                 scheduler: torch.optim.lr_scheduler._LRScheduler, 
                 criterion: torch.nn.Module) -> None:
        """
        Initialize the ModelTrainer class.

        :param ml_model: The machine learning model to be trained.
        :type ml_model: torch.nn.Module
        :param optimizer: The optimizer for training the model.
        :type optimizer: torch.optim.Optimizer
        :param scheduler: The learning rate scheduler.
        :type scheduler: torch.optim.lr_scheduler._LRScheduler
        :param criterion: The loss function.
        :type criterion: torch.nn.Module
        """
        self.device = torch.device("cuda" if torch.cuda.is_available() else 'cpu')
        self.ml_model = ml_model.to(self.device)
        self.optimizer = optimizer
        self.scheduler = scheduler
        self.criterion = criterion
        

    def prepare_batch(self, batch: dict) -> Tuple[torch.Tensor, torch.Tensor]:
        """
        Prepare a batch of data for training or validation.

        :param batch: A batch of data.
        :type batch: dict
        :return: A tuple containing inputs and labels.
        :rtype: Tuple[torch.Tensor, torch.Tensor]
        """
        inputs = batch['embedding'].to(self.device)
        labels = batch['label'].float().to(self.device)
        
        # Ensure labels are two-dimensional
        if labels.dim() == 1:
            labels = labels.unsqueeze(1)
        return inputs, labels
    
    # def eval_metric_function(self, labels, preds):
    #     unique_classes = np.unique(labels.cpu().numpy())
    #     if len(unique_classes) == 1:
    #         return 0 # or return 0 if you prefer
    #     eval_metric_func = getattr(skmetrics, self.eval_metric)
    #     return eval_metric_func(labels.to(torch.int32).flatten().tolist(), preds.flatten().tolist())
        
    def train_epoch(self, data_loader: torch.utils.data.DataLoader) -> float:
        """
        Train the model for one epoch.

        :param data_loader: DataLoader for the training data.
        :type data_loader: torch.utils.data.DataLoader
        :return: The average training loss for the epoch.
        :rtype: float
        """
        self.ml_model.train()
        total_loss = 0
        total_metric = 0
        for batch in data_loader:
            self.optimizer.zero_grad()
            inputs, labels = self.prepare_batch(batch)

            outputs = self.ml_model(inputs)
            loss = self.criterion(outputs, labels)
            loss.backward()
            self.optimizer.step()
            self.scheduler.step()

            total_loss += loss.item()

        avg_loss = total_loss / len(data_loader)

        return avg_loss

    def validate_epoch(self, data_loader: torch.utils.data.DataLoader) -> Tuple[float, List[int], List[float]]:
        """
        Validate the model for one epoch.

        :param data_loader: DataLoader for the validation data.
        :type data_loader: torch.utils.data.DataLoader
        :return: A tuple containing the average validation loss, labels, and predictions for the epoch.
        :rtype: Tuple[float, List[int], List[float]]
        """
        self.ml_model.eval()
        total_loss = 0
        total_metric = 0
        labels_per_epoch = []
        preds_per_epoch = []
        with torch.no_grad():
            for batch in data_loader:
                inputs, labels = self.prepare_batch(batch)
                outputs = self.ml_model(inputs)
                loss = self.criterion(outputs, labels)
                total_loss += loss.item()
                labels_per_epoch.extend(labels.to(torch.int32).flatten().tolist())
                preds_per_epoch.extend(outputs.flatten().tolist())
        
        avg_loss = total_loss / len(data_loader)
        return avg_loss, labels_per_epoch, preds_per_epoch

    def train_epochs(self, 
                     train_dataloader: torch.utils.data.DataLoader, 
                     val_dataloader: torch.utils.data.DataLoader, 
                     num_epochs: int = 10) -> Tuple[List[float], List[float], List[List[int]], List[List[float]]]:
        """
        Train and validate the model for a specified number of epochs.

        :param train_dataloader: DataLoader for the training data.
        :type train_dataloader: torch.utils.data.DataLoader
        :param val_dataloader: DataLoader for the validation data.
        :type val_dataloader: torch.utils.data.DataLoader
        :param num_epochs: Number of epochs to train the model, defaults to 10
        :type num_epochs: int, optional
        :return: A tuple containing average training loss, average validation loss, labels, and predictions for each epoch.
        :rtype: Tuple[List[float], List[float], List[List[int]], List[List[float]]]
        """

        avg_train_loss = []
        avg_val_loss = []
        epochs_preds = []
        epochs_labels = []
        self.ml_model = self.ml_model.to(self.device)
        for epoch in tqdm(range(num_epochs), desc="Epochs"):
            train_loss = self.train_epoch(train_dataloader)
            val_loss, val_labels, val_preds = self.validate_epoch(val_dataloader)

            # Update each result
            avg_train_loss.append(train_loss)
            avg_val_loss.append(val_loss)
            epochs_labels.append(val_labels)
            epochs_preds.append(val_preds)
            
        return avg_train_loss, avg_val_loss, epochs_labels, epochs_preds

#### ModelTrainer test cases

In [10]:
import unittest
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, Dataset
from typing import List, Tuple, Any
from tqdm import tqdm

class DummyDataset(Dataset):
    def __init__(self, size: int):
        self.size = size
        self.data = torch.randn(size, 10)
        self.labels = torch.randint(0, 2, (size, 1)).float()

    def __len__(self):
        return self.size

    def __getitem__(self, idx: int):
        return {'embedding': self.data[idx], 'label': self.labels[idx]}

class DummyModel(nn.Module):
    def __init__(self, input_size: int):
        super(DummyModel, self).__init__()
        self.linear = nn.Linear(input_size, 1)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return torch.sigmoid(self.linear(x))

class TestModelTrainer(unittest.TestCase):

    def setUp(self):
        self.train_dataset = DummyDataset(size=100)
        self.val_dataset = DummyDataset(size=50)
        self.train_dataloader = DataLoader(self.train_dataset, batch_size=16, shuffle=True)
        self.val_dataloader = DataLoader(self.val_dataset, batch_size=16, shuffle=False)
        self.ml_model = DummyModel(input_size=10)
        self.optimizer = torch.optim.Adam(self.ml_model.parameters(), lr=0.001)
        self.scheduler = torch.optim.lr_scheduler.StepLR(self.optimizer, step_size=1, gamma=0.1)
        self.criterion = nn.BCELoss()
        self.trainer = ModelTrainer(ml_model=self.ml_model, optimizer=self.optimizer, scheduler=self.scheduler, criterion=self.criterion)

    def test_initialization(self):
        self.assertEqual(self.trainer.ml_model, self.ml_model)
        self.assertEqual(self.trainer.optimizer, self.optimizer)
        self.assertEqual(self.trainer.scheduler, self.scheduler)
        self.assertEqual(self.trainer.criterion, self.criterion)
        self.assertTrue(isinstance(self.trainer.device, torch.device))

    def test_prepare_batch(self):
        batch = next(iter(self.train_dataloader))
        inputs, labels = self.trainer.prepare_batch(batch)
        self.assertEqual(inputs.shape, (16, 10))
        self.assertEqual(labels.shape, (16, 1))

    def test_train_epoch(self):
        avg_loss = self.trainer.train_epoch(self.train_dataloader)
        self.assertIsInstance(avg_loss, float)

    def test_validate_epoch(self):
        avg_loss, labels, preds = self.trainer.validate_epoch(self.val_dataloader)
        self.assertIsInstance(avg_loss, float)
        self.assertIsInstance(labels, list)
        self.assertIsInstance(preds, list)
        self.assertEqual(len(labels), len(self.val_dataset))
        self.assertEqual(len(preds), len(self.val_dataset))

    def test_train_epochs(self):
        avg_train_loss, avg_val_loss, epochs_labels, epochs_preds = self.trainer.train_epochs(self.train_dataloader, self.val_dataloader, num_epochs=2)
        self.assertIsInstance(avg_train_loss, list)
        self.assertIsInstance(avg_val_loss, list)
        self.assertIsInstance(epochs_labels, list)
        self.assertIsInstance(epochs_preds, list)
        self.assertEqual(len(avg_train_loss), 2)
        self.assertEqual(len(avg_val_loss), 2)
        self.assertEqual(len(epochs_labels), 2)
        self.assertEqual(len(epochs_preds), 2)
unittest.main(argv=['first-arg-is-ignored'], exit=False)

Epochs: 100%|██████████| 2/2 [00:00<00:00, 87.02it/s]
..
----------------------------------------------------------------------
Ran 5 tests in 0.784s

OK


### Hyperparameter tuning module

In [6]:
from ray.tune.search.bayesopt import BayesOptSearch
from ray.tune.search.hyperopt import HyperOptSearch
import torch
import torch.nn as nn
from torch.utils.data import DataLoader
from functools import partial
from ray import tune
from ray.tune import CLIReporter
from ray.tune.schedulers import ASHAScheduler
from typing import List, Dict, Any, Optional

In [13]:
class HyperparameterTuner:
    """ Hyperparameter tune the model
    """
    def __init__(self, 
                 train_dataset: torch.utils.data.Dataset,
                 val_dataset: torch.utils.data.Dataset) -> None:
        """
        Initialize the HyperparameterTuner class.

        :param train_dataset: The training dataset.
        :type train_dataset: torch.utils.data.Dataset
        :param val_dataset: The validation dataset.
        :type val_dataset: torch.utils.data.Dataset
        :param ml_model: The machine learning model to be tuned.
        :type ml_model: torch.nn.Module
        """
        self.train_dataset = train_dataset
        self.val_dataset = val_dataset
    
    def hptune(self, 
               param_space: Dict[str, Any],
               search_algo = 'tpe',
               verbose: int = 1,
               threshold: float = 0.5,
               num_trials: int = 10) -> Any:
        """
        Perform hyperparameter tuning.

        :param param_space: The parameter space for tuning.
        :type param_space: Dict[str, Any]
        :param eval_metrics: List of evaluation metrics.
        :type eval_metrics: List[str]
        :param search_algo: Hyperparameter tuning search algorithm, defaults tp Tree-Structured Parzen Estimator (TPE);
                            TPE is Bayesian optimization algorithm by by J. Bergstra and colleagues
        :type algo_algo: str, optional
        :param verbose: Verbosity level, defaults to 1
        :type verbose: int, optional
        :param threshold: Threshold for evaluation, defaults to 0.5
        :type threshold: float, optional
        :param num_trials: Number of samples for tuning, defaults to 10
        :type num_trials: int, optional
        :return: The result grid of the tuning process.
        :rtype: Any
        """

        # trainable = partial(
        #     hp_train_func, 
        #     train_dataset = self.train_dataset,
        #     val_dataset = self.val_dataset,
        #     verbose = verbose
        # )
        
        self._configure_tune(search_algo, num_trials)
        self._configure_run()
        trainable_with_resources = tune.with_resources(hp_train_func, {"cpu": 1})
        tuner = tune.Tuner(
            tune.with_parameters(trainable_with_resources, train_dataset = self.train_dataset,
                                                        val_dataset = self.val_dataset,
                                                        verbose = verbose,
                                                        threshold = threshold),
            param_space = param_space,
            run_config = self.run_config,
            tune_config = self.tune_config
        )
        result_grid = tuner.fit()
        return result_grid
        
    def _configure_tune(self, search_algo, num_trials):
        """
        Configure the hyperparameter tuning process.
        
        :param search_algo: Hyperparameter tuning search algorithm, defaults tp Tree-Structured Parzen Estimator (TPE);
                            TPE is Bayesian optimization algorithm by by J. Bergstra and colleagues
        :type algo_algo: str, optional        
        :param num_trials: Number of tuning trials.
        :type num_trials: int
        """
        if search_algo == 'random': 
            pass
        else: 
            search_alg = HyperOptSearch(metric="val_loss", mode="min")
        self.tune_config = tune.TuneConfig(
            search_alg = search_alg,
            mode='min', 
            metric = 'val_loss',  # this metric is the the one reported by train report function
            num_samples=num_trials
        )
    def _configure_run(self) -> None:
        """
        Configure the hyperparameter tuning process.

        :param num_trials: Number of samples for tuning.
        :type num_trials: int
        """
        self.run_config = train.RunConfig(
            progress_reporter=ExperimentTerminationReporter(),
            stop = {"val_loss": 0.2}
        )

class ExperimentTerminationReporter(CLIReporter):
    def should_report(self, trials, done=True):
        """Reports only on experiment termination."""
        return done



def hp_train_func(config: Dict[str, Any], 
                  train_dataset: torch.utils.data.Dataset, 
                  val_dataset: torch.utils.data.Dataset, 
                  verbose: int,
                  threshold: float = 0.5) -> Dict[str, float]:
    """
    Training function for hyperparameter tuning.

    :param config: Configuration dictionary.
    :type config: Dict[str, Any]
    :param train_dataset: The training dataset.
    :type train_dataset: torch.utils.data.Dataset
    :param val_dataset: The validation dataset.
    :type val_dataset: torch.utils.data.Dataset
    :param ml_model: The machine learning model to be trained.
    :type ml_model: nn.Module
    :param eval_metrics: List of evaluation metrics.
    :type eval_metrics: List[str]
    :param verbose: Verbosity level.
    :type verbose: int
    :param threshold: Threshold for evaluation, defaults to 0.5
    :type threshold: float, optional
    :return: Evaluation results.
    :rtype: Dict[str, float]
    """
    
    
    
    train_dataloader = DataLoader(train_dataset, batch_size=config['batch_size'], shuffle=True)
    val_dataloader = DataLoader(val_dataset, batch_size=config['batch_size'], shuffle=False)

    # reset model architecture, hidden size and dropout_rate
    ml_model = DNNClassifier(input_size = train_dataset.input_size)
    ml_model.set_architecture(hidden_sizes = config['hidden_sizes'],
                           dropout_rate = config['dropout_rate'])
    device = torch.device("cuda" if torch.cuda.is_available() else 'cpu')
    ml_model.to(device)
    
    # tune different types of optimizer and learning rate
    optimizer = _get_optimizer(config, ml_model)
        
    total_steps = len(train_dataloader) * config['num_epochs']
    scheduler = get_linear_schedule_with_warmup(optimizer, 
                                                num_warmup_steps=0, 
                                                num_training_steps=total_steps)
    criterion = nn.BCELoss()
    model_trainer = ModelTrainer(ml_model = ml_model, 
                                 optimizer = optimizer,
                                  scheduler = scheduler,
                                  criterion = criterion)
    
    # check the trainer class and maybe implement another function that only return result_per_epoch; # still use 0.5 to hyperparaemter tuning
    avg_train_loss, avg_val_loss, epochs_labels, epochs_preds_scores = model_trainer.train_epochs(train_dataloader, 
                                                                                             val_dataloader, 
                                                                                             num_epochs = config['num_epochs'])
    # report avg. val loss and specified metrics across num_epochs
    hp_eval_results = {'val_loss' : sum(avg_val_loss)/len(avg_val_loss)}
    
    if verbose > 0:
        train.report(hp_eval_results)
    return hp_eval_results

def _get_optimizer(config: Dict[str, Any], ml_model: nn.Module) -> torch.optim.Optimizer:
    """
    Get the optimizer based on the configuration.

    :param config: Configuration dictionary.
    :type config: Dict[str, Any]
    :param ml_model: The machine learning model.
    :type ml_model: nn.Module
    :return: The optimizer.
    :rtype: torch.optim.Optimizer
    """

    if config['optimizer'] == 'adam': 
        return torch.optim.AdamW(ml_model.parameters(), lr=config['lr'])
    elif config['optimizer'] == 'sgd' and 'momentum' in config:
        return torch.optim.SGD(ml_model.parameters(), lr=config['lr'], momentum=config['momentum'])

#### Hyperparameter tuning test module

In [21]:
import unittest
from unittest.mock import MagicMock, patch
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, Dataset
from ray import tune
from ray.tune.search.hyperopt import HyperOptSearch
from ray.tune import CLIReporter
from functools import partial
from typing import List, Dict, Any

# Assuming the classes and functions are defined in a module named `hyperparameter_tuning`

class DummyDataset(Dataset):
    def __init__(self, size: int):
        self.size = size
        self.data = torch.randn(size, 10)
        self.labels = torch.randint(0, 2, (size, 1)).float()

    def __len__(self):
        return self.size

    def __getitem__(self, idx: int):
        return {'embedding': self.data[idx], 'label': self.labels[idx]}

class TestHyperparameterTuner(unittest.TestCase):

    def setUp(self):
        self.train_dataset = DummyDataset(size=100)
        self.val_dataset = DummyDataset(size=50)
        self.ml_model = DNNClassifier(input_size=10)
        self.tuner = HyperparameterTuner(train_dataset=self.train_dataset, val_dataset=self.val_dataset, ml_model=self.ml_model)

    def test_initialization(self):
        self.assertEqual(self.tuner.train_dataset, self.train_dataset)
        self.assertEqual(self.tuner.val_dataset, self.val_dataset)
        self.assertEqual(self.tuner.ml_model, self.ml_model)

    def test_hptune(self):
        param_space = {
            'hidden_sizes': [64, 32],
            'dropout_rate': 0.3,
            'batch_size': 16,
            'num_epochs': 5,
            'optimizer': 'adam',
            'lr': 0.001
        }
        eval_metrics = ['accuracy']

        # Mocking the Tuner and its fit method
        with unittest.mock.patch('ray.tune.Tuner') as MockTuner:
            mock_tuner_instance = MockTuner.return_value
            mock_tuner_instance.fit.return_value = 'result_grid'

            result = self.tuner.hptune(param_space=param_space, eval_metrics=eval_metrics, num_trials=5)

            self.assertEqual(result, 'result_grid')
            MockTuner.assert_called_once()

    def test_get_optimizer(self):
        config = {
            'optimizer': 'adam',
            'lr': 0.001
        }
        optimizer = _get_optimizer(config, self.ml_model)
        self.assertIsInstance(optimizer, torch.optim.AdamW)

        config = {
            'optimizer': 'sgd',
            'lr': 0.01,
            'momentum': 0.9
        }
        optimizer = _get_optimizer(config, self.ml_model)
        self.assertIsInstance(optimizer, torch.optim.SGD)

class TestExperimentTerminationReporter(unittest.TestCase):

    def test_should_report(self):
        reporter = ExperimentTerminationReporter()
        self.assertTrue(reporter.should_report(trials=[], done=True))
        self.assertFalse(reporter.should_report(trials=[], done=False))

class TestHpTrainFunc(unittest.TestCase):

    def test_hp_train_func(self):
        config = {
            'hidden_sizes': [64, 32],
            'dropout_rate': 0.3,
            'batch_size': 16,
            'num_epochs': 5,
            'optimizer': 'adam',
            'lr': 0.001
        }
        train_dataset = DummyDataset(size=100)
        val_dataset = DummyDataset(size=50)
        ml_model = DNNClassifier(input_size=10)
        eval_metrics = ['lift']
        verbose = 1

        result = hp_train_func(config, train_dataset, val_dataset, ml_model, eval_metrics, verbose)

        self.assertIn('val_loss', result)
        self.assertIsInstance(result['val_loss'], float)
unittest.main(argv=['first-arg-is-ignored'], exit=False)

Epochs: 100%|██████████| 2/2 [00:00<00:00, 94.00it/s]
..
----------------------------------------------------------------------
Ran 10 tests in 0.129s

OK


In [30]:
!pip install hyperopt

### Evaluation module

In this class, the evaluator is implementated using Classification model module; this will performs the following function
* split (texts, labels)
* encode each set text to embedding
* call classification models and perform (fit, tune, evaluate) function
* return the best ML model for the specified embedding model on the test set, return score

In [14]:
from imblearn.under_sampling import RandomUnderSampler
from imblearn.over_sampling import RandomOverSampler
from imblearn.combine import SMOTEENN

In [15]:
from functools import wraps
from inspect import signature

def filter_kwargs(func):
    """
    Define a decorator that filters **kwargs based on the function's signature.
    This approach automatically removes any keyword arguments that the function does not accept.

    :param func: The function to be decorated.
    :type func: Callable
    :return: The decorated function with filtered keyword arguments.
    :rtype: Callable
    """
    @wraps(func)
    def wrapper(**kwargs):
        sig = signature(func)
        valid_keys = sig.parameters.keys()
        filtered_kwargs = {k: v for k, v in kwargs.items() if k in valid_keys}
        return func(**filtered_kwargs)
    return wrapper


def split_data(X: np.ndarray, 
               y: np.ndarray, 
               test_size: float = 0.1, 
               val_size: float = 0.1, 
               **kwargs: Any) -> Tuple[np.ndarray, np.ndarray, np.ndarray, np.ndarray, np.ndarray, np.ndarray]:
    
    """
    Splits data into training, testing, and optionally validation sets.

    :param X: Features dataset.
    :type X: np.ndarray
    :param y: Target dataset.
    :type y: np.ndarray
    :param test_size: Proportion of the dataset to include in the test split, defaults to 0.1.
    :type test_size: float, optional
    :param val_size: Proportion of the dataset to include in the validation split, defaults to 0.1.
    :type val_size: float, optional
    :param kwargs: Additional keyword arguments to be passed to train_test_split function.
    :type kwargs: Any
    :return: A tuple containing the split datasets: (X_train, X_val, X_test, y_train, y_val, y_test).
    :rtype: Tuple[np.ndarray, np.ndarray, np.ndarray, np.ndarray, np.ndarray, np.ndarray]
    """
    
    # Adjust test_size for initial split to account for subsequent validation split
    initial_test_size = test_size / (1 - val_size) if val_size + test_size < 1 else test_size
    X_temp, X_test, y_temp, y_test = train_test_split(X, y, test_size=initial_test_size, stratify = y, **kwargs)
    # Split the temporary training set into final training and validation sets
    X_train, X_val, y_train, y_val = train_test_split(X_temp, y_temp, test_size=val_size / (1 - initial_test_size), stratify = y_temp, **kwargs)
    return X_train, X_val, X_test, y_train, y_val, y_test

In [16]:
class Evaluator(ABC):
    """Base class for all evaluators
    Extend this class and implement perform_evaluation for custom evaluators.
    """

    def __init__(self, 
                 seed: int = 42, 
                 **kwargs):
        self.seed = seed
        random.seed(self.seed)
        np.random.seed(self.seed)
        torch.manual_seed(self.seed)
        torch.cuda.manual_seed_all(self.seed)
        
    @abstractmethod
    def _prepare_datasets(self):
        pass

    @abstractmethod
    def _tune_hyperparameters(self):
        pass

    @abstractmethod
    def _train_model(self):
        pass

    @abstractmethod
    def _predict(self):
        pass
    
    @abstractmethod
    def perform_evaluation(self):
        pass

class XgboostEvaluator(Evaluator):
    pass

class DNNEvaluator(Evaluator):

    def __init__(self,
                 embeddings: np.ndarray,
                 labels: np.ndarray, 
                 rebalance: Union[bool, str] = False,
                 test_size: float = 0.1, 
                 val_size: float = 0.1,
                 **kwargs: Any) -> None:
        """
        Initialize the DNNEvaluator class.

        :param embeddings: The embeddings dataset.
        :type embeddings: np.ndarray
        :param labels: The labels dataset.
        :type labels: np.ndarray
        :param rebalance: Whether to rebalance the dataset, defaults to False.
                          If a string is provided, it should be the class name of the resampling method from imbalanced-learn.
        :type rebalance: Union[bool, str]
        :param kwargs: Additional keyword arguments.
        """
        super().__init__(**kwargs)
        self.classifier = None
        self.best_config = None
        self._prepare_datasets(embeddings, 
                               labels, 
                               rebalance = rebalance) # Add paramss
        self.device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
        self.model_trained = False

    def _rebalance_dataset(self, 
                           X: np.ndarray, 
                           y: np.ndarray,
                           rebalance: Union[str, bool]) -> Tuple[np.ndarray, np.ndarray]:
        """
        Rebalance the dataset using RandomOverSampler.

        :param X: Features dataset.
        :type X: np.ndarray
        :param y: Labels dataset.
        :type y: np.ndarray
        :return: Resampled features and labels.
        :rtype: Tuple[np.ndarray, np.ndarray]
        """

        # Dynamically import the resampling class from imbalanced-learn
        try: 
            if rebalance == 'ros':
                sampler = RandomOverSampler(random_state = 44)
            elif rebalance == 'rus':
                sampler = RandomUnderSampler(random_state = 44)
            elif rebalance == 'smoteenn':
                sampler = SMOTEENN(random_state = 44)
        except (ImportError, AttributeError) as e:
            raise ValueError(f"Error importing resampling method {rebalance}: {e} - ros, rus, smoteenn")
        
        X_resampled, y_resampled = sampler.fit_resample(X, y)
        return X_resampled, y_resampled
    
    def _prepare_datasets(self, 
                          embeddings: np.ndarray, 
                          labels: np.ndarray, 
                          rebalance: bool,
                          test_size: float = 0.1, 
                         val_size: float = 0.1,
                          **kwargs: Any) -> None:
        """
        Prepare the datasets for training, validation, and testing.

        :param embeddings: The embeddings dataset.
        :type embeddings: np.ndarray
        :param labels: The labels dataset.
        :type labels: np.ndarray
        :param rebalance: Whether to rebalance the dataset.
        :type rebalance: bool
        :param kwargs: Additional keyword arguments.
        """
    
        embedding_train, embedding_val, embedding_test, y_train, y_val, y_test = split_data(embeddings, 
                                                                                            labels, 
                                                                                            test_size = test_size,
                                                                                            val_size = val_size,
                                                                                            **kwargs)
        if rebalance:
            embedding_train, y_train = self._rebalance_dataset(embedding_train, y_train, rebalance)
            
        self.train_dataset = EmbeddingDatasets(embedding_train.astype('float32'), y_train)
        self.val_dataset = EmbeddingDatasets(embedding_val.astype('float32'), y_val)
        self.test_dataset = EmbeddingDatasets(embedding_test.astype('float32'), y_test)

    def _tune_hyperparameters(self, 
                              param_space: Dict[str, Any],
                              num_trials: int,
                              threshold: float,
                              verbose: int) -> Any:
        """
        Tune the hyperparameters of the model.

        :param param_space: The parameter space for hyperparameter tuning.
        :type param_space: dict
        :param eval_metrics: The evaluation metrics.
        :type eval_metrics: list
        :param num_trials: The number of samples for hyperparameter tuning.
        :type num_trials: int
        :param threshold: The threshold for evaluation metrics.
        :type threshold: float
        :param verbose: The verbosity level.
        :type verbose: int
        :return: The results of hyperparameter tuning.
        :rtype: Any
        """
        
        hptuner = HyperparameterTuner(self.train_dataset, 
                                      self.val_dataset)
        grid_results = hptuner.hptune(param_space = param_space, 
                                      num_trials = num_trials,
                                      verbose = verbose,
                                      threshold = threshold)
        # turn off the model access during hyperparameter tuning, not trained 
        self.model_trained = False
        return grid_results
        
    def _train_model(self, 
                     train_dataloader: DataLoader, 
                     val_dataloader: DataLoader,
                     config: Dict[str, Any]) -> Tuple[List[float], List[float], List[List[int]], List[List[float]]]:
        """
        Train the model.

        :param train_dataloader: DataLoader for the training data.
        :type train_dataloader: DataLoader
        :param val_dataloader: DataLoader for the validation data.
        :type val_dataloader: DataLoader
        :param config: The configuration for training.
        :type config: dict
        :return: Training and validation losses, labels, and predictions for each epoch.
        :rtype: Tuple[List[float], List[float], List[List[int]], List[List[float]]]
        """
        
        total_steps = len(train_dataloader) * config['num_epochs']

        # get the best optimzer based on  the best config 
        optimizer = _get_optimizer(config = config, 
                                   ml_model = self.classifier)
        
        scheduler = get_linear_schedule_with_warmup(optimizer, 
                                                    num_warmup_steps=0, 
                                                    num_training_steps=total_steps)
        criterion = nn.BCELoss()

        model_trainer = ModelTrainer(ml_model = self.classifier, 
                                         optimizer = optimizer,
                                          scheduler = scheduler,
                                          criterion = criterion)
        avg_train_loss, avg_val_loss, epochs_labels, epochs_preds = model_trainer.train_epochs(train_dataloader, 
                                                                                                 val_dataloader,
                                                                                                 num_epochs = config['num_epochs'])
        self.model_trained = True
        return avg_train_loss, avg_val_loss, epochs_labels, epochs_preds

    def _predict(self, 
                 test_dataloader: DataLoader) -> Tuple[List[int], List[float]]:
        """
        Make predictions using the trained model.

        :param test_dataloader: DataLoader for the test data.
        :type test_dataloader: DataLoader
        :return: Test labels and predictions.
        :rtype: Tuple[List[int], List[float]]
        """
        
        self.classifier.eval()
        test_labels = []
        test_preds = []
        with torch.no_grad():
            for batch in test_dataloader:
                inputs = batch['embedding'].to(self.device)
                labels = batch['label'].unsqueeze(1).float().to(self.device)
                outputs = self.classifier(inputs)
                test_labels.extend(labels.to(torch.int32).flatten().tolist())
                test_preds.extend(outputs.flatten().tolist())
        return test_labels, test_preds     
    
    
    def _fit_tune_model(self, 
                        param_space: Dict[str, Any], 
                        threshold: float,
                        tuning_verbose: int,
                        num_samples: int):
        
        pass
    
    
    def predict_with_best(self, test_data):
        # here check the datatype dataframe, iters, numpy
        test_torch = torch.from_numpy(test_data)
        pass
    
    
    
    def get_best_model(self) -> DNNClassifier:
        """
        Get the best model based on hyperparameter tuning.

        :return: The best model.
        :rtype: DNNClassifier
        :raises ValueError: If no best model is available.
        """
        if self.model_trained and self.best_config:
            return self.classifier
        raise ValueError("No best model is available")
        
    def save_model(self, file_path):
        
        best_model = self.get_best_model()
        torch.save(best_model.state_dict(), filepath)
        print(f"Save model to {filepath}")
    
    
    def perform_evaluation(self, 
                           param_space: Dict[str, Any],
                           eval_metrics: List[str], 
                           threshold: float = 0.5,
                           tuning_verbose: int = 0,
                           num_trials: int = 10) -> Tuple[Dict[str, Any], Dict[str, Any]]:
        """
        Perform the evaluation process.

        :param param_space: The parameter space for hyperparameter tuning.
        :type param_space: dict
        :param eval_metrics: The evaluation metrics.
        :type eval_metrics: list
        :param threshold: The threshold for evaluation metrics, defaults to 0.35.
        :type threshold: float
        :param tuning_verbose: The verbosity level for hyperparameter tuning, defaults to 0.
        :type tuning_verbose: int
        :param num_trials: The number of samples for hyperparameter tuning, defaults to 10.
        :type num_trials: int
        :return: Validation and test metrics.
        :rtype: Tuple[dict, dict]
        """
        
        
        
        # 1. hyperparameter tuning and get the best configurations
        # Refactor this part; separate the hp from teh model construction in this class; hyperparamter has a indepdnent model, not self.classificer
        hp_results = self._tune_hyperparameters(param_space = param_space,
                                                   verbose = tuning_verbose,
                                                   threshold = threshold,
                                                   num_trials = num_trials)
        self.best_config = hp_results.get_best_result().config

        # 2. set up the ml model with the best config
        input_size = self.train_dataset.input_size
        self.classifier = DNNClassifier(input_size = input_size)
        self.classifier.set_architecture(self.best_config['hidden_sizes'],
                                         self.best_config['dropout_rate'])
        
        # 3. create dataloaders that fit the corresponding ml model
        train_dataloader = DataLoader(self.train_dataset, batch_size=self.best_config['batch_size'], shuffle=True)
        val_dataloader = DataLoader(self.val_dataset, batch_size=self.best_config['batch_size'], shuffle=False)
        test_dataloader = DataLoader(self.test_dataset, batch_size=self.best_config['batch_size'], shuffle=False)

        # 4. train and validate with the best model; TODO: free the setting of hp spaces where it's able to identify the parameters and only adjust the specified parameters
        avg_train_loss, avg_val_loss, epochs_labels, epochs_preds_scores = self._train_model(train_dataloader = train_dataloader, 
                                                                                            val_dataloader = val_dataloader, 
                                                                                            config = self.best_config)
        validate_metrics = evaluate_scores_epochs(epochs_labels, 
                                                  epochs_preds_scores, 
                                                  eval_metrics,
                                                  threshold = threshold)
        # 5. predict using test set
        test_labels, test_preds_scores = self._predict(test_dataloader)
        test_metrics = evaluate_scores(test_labels, 
                                       test_preds_scores, 
                                       eval_metrics,
                                      threshold = threshold)
        return validate_metrics, test_metrics

#### Evaluator test module

In [24]:
import unittest
import torch
import numpy as np

class TestDNNEvaluator(unittest.TestCase):
    def setUp(self):
        # Create mock data
        self.embeddings = np.random.rand(100, 10).astype('float32')
        self.labels = np.random.randint(0, 2, 100).astype('float32')

        # Patch the split_data function
        global split_data_func
        split_data_func = split_data

        # Initialize DNNEvaluator
        self.evaluator = DNNEvaluator(embeddings=self.embeddings, labels=self.labels)

    def test_prepare_datasets(self):
        self.assertEqual(len(self.evaluator.train_dataset), 78)
        self.assertEqual(len(self.evaluator.val_dataset), 10)
        self.assertEqual(len(self.evaluator.test_dataset), 12)

    def test_perform_evaluation(self):
        param_space = {
            'hidden_sizes': tune.choice([[128, 64], [256, 128], [512, 256]]),
            'dropout_rate': tune.uniform(0.1, 0.5),
            'batch_size': tune.choice([16, 32, 64]),
            'lr': tune.loguniform(1e-5, 1e-3),
            'optimizer': tune.choice(['adam', 'sgd']),
            'num_epochs': tune.choice([10, 20, 30]),
            'momentum': tune.uniform(0.90, 0.99)
        }
        eval_metrics = [
                         'average_precision_score', 
                         'roc_auc_score', 'precision_score', 
                         'recall_score', 
                         'f1_score', 
                         'accuracy_score']
        val_metrics_per_epoch, test_metrics = self.evaluator.perform_evaluation(eval_metrics=eval_metrics, param_space = param_space, num_trials=2)
        self.assertIsInstance(val_metrics_per_epoch, dict)
        self.assertIsInstance(test_metrics, dict)
        for metric in eval_metrics:
            self.assertIn(metric, test_metrics)
unittest.main(argv=['first-arg-is-ignored'], exit=False)

/opt/conda/envs/pytorch/lib/python3.10/site-packages/ray/_private/node.py:1350: ResourceWarning: unclosed file <_io.TextIOWrapper name='/var/tmp/ray/session_2024-07-31_02-55-01_022074_3214/logs/gcs_server.out' mode='a' encoding='utf-8'>
  self.start_gcs_server()
/opt/conda/envs/pytorch/lib/python3.10/site-packages/ray/_private/node.py:1350: ResourceWarning: unclosed file <_io.TextIOWrapper name='/var/tmp/ray/session_2024-07-31_02-55-01_022074_3214/logs/gcs_server.err' mode='a' encoding='utf-8'>
  self.start_gcs_server()
/opt/conda/envs/pytorch/lib/python3.10/site-packages/ray/_private/node.py:1355: ResourceWarning: unclosed file <_io.TextIOWrapper name='/var/tmp/ray/session_2024-07-31_02-55-01_022074_3214/logs/monitor.out' mode='a' encoding='utf-8'>
  self.start_monitor()
/opt/conda/envs/pytorch/lib/python3.10/site-packages/ray/_private/node.py:1355: ResourceWarning: unclosed file <_io.TextIOWrapper name='/var/tmp/ray/session_2024-07-31_02-55-01_022074_3214/logs/monitor.err' mode='a' e

== Status ==
Current time: 2024-07-31 02:55:18 (running for 00:00:09.29)
Using FIFO scheduling algorithm.
Logical resource usage: 1.0/8 CPUs, 0/1 GPUs (0.0/1.0 accelerator_type:T4)
Current best trial: e78828c5 with val_loss=0.6856654147307079 and parameters={'hidden_sizes': (256, 128), 'dropout_rate': 0.45188377668119795, 'batch_size': 32, 'lr': 5.540671882321977e-05, 'optimizer': 'sgd', 'num_epochs': 30, 'momentum': 0.9704793353308432}
Result logdir: /var/tmp/ray/session_2024-07-31_02-55-01_022074_3214/artifacts/2024-07-31_02-55-09/hp_train_func_2024-07-31_02-55-00/driver_artifacts
Number of trials: 2/2 (2 TERMINATED)
+------------------------+------------+-------------------+--------------+----------------+----------------+-------------+------------+--------------+-------------+--------+------------------+------------+
| Trial name             | status     | loc               |   batch_size |   dropout_rate | hidden_sizes   |          lr |   momentum |   num_epochs | optimizer   |   

Epochs: 100%|██████████| 2/2 [00:00<00:00, 93.20it/s]
..
----------------------------------------------------------------------
Ran 12 tests in 19.756s

OK


#### Evaluator test case

In [33]:
embeddings = np.random.rand(300, 10).astype('float32')
labels = np.random.randint(0, 2, 300).astype('float32')
param_space = {
    'hidden_sizes': tune.choice([[128, 64], [256, 128], [512, 256]]),
    'dropout_rate': tune.uniform(0.1, 0.5),
    'batch_size': tune.choice([16, 32, 64]),
    'lr': tune.loguniform(1e-3, 1e-1),
    'optimizer': tune.choice(['adam', 'sgd']),
    'num_epochs': tune.choice([10, 20, 30]),
    'momentum': tune.uniform(0.90, 0.99)
}
eval_metrics = ['confusion_matrix', 
                 'average_precision_score', 
                 'roc_auc_score', 
                'precision_score', 
                 'recall_score', 
                 'f1_score', 
                 'accuracy_score']


In [34]:
evaluator = DNNEvaluator(embeddings=embeddings, 
                         labels=labels)
val_metrics, test_metrics = evaluator.perform_evaluation(eval_metrics=eval_metrics, 
                             param_space = param_space,
                             num_trials=1)

2024-07-12 23:36:50,781	INFO tune.py:616 -- [output] This uses the legacy output and progress reporter, as Jupyter notebooks are not supported by the new engine, yet. For more information, please see https://github.com/ray-project/ray/issues/36949
2024-07-12 23:37:39,895	INFO tensorboardx.py:308 -- Removed the following hyperparameter values when logging to tensorboard: {'hidden_sizes': (256, 128)}
2024-07-12 23:37:39,971	INFO tune.py:1009 -- Wrote the latest version of all result files and experiment state to '/home/a964286/ray_results/hp_train_func_2024-07-12_23-36-50' in 0.0662s.
2024-07-12 23:37:40,033	INFO tune.py:1041 -- Total run time: 49.25 seconds (48.04 seconds for the tuning loop).


== Status ==
Current time: 2024-07-12 23:37:40 (running for 00:00:48.16)
Using FIFO scheduling algorithm.
Logical resource usage: 1.0/8 CPUs, 0/0 GPUs
Current best trial: 6ea06077 with val_loss=0.7053296864032745 and parameters={'hidden_sizes': (256, 128), 'dropout_rate': 0.17897507927017978, 'batch_size': 16, 'lr': 0.06866572012820302, 'optimizer': 'sgd', 'num_epochs': 10, 'momentum': 0.911423590684937}
Result logdir: /tmp/ray/session_2024-07-12_23-30-25_595554_13474/artifacts/2024-07-12_23-36-51/hp_train_func_2024-07-12_23-36-50/driver_artifacts
Number of trials: 1/1 (1 TERMINATED)




Epochs:   0%|          | 0/10 [00:00<?, ?it/s]

In [45]:
pd.DataFrame(val_metrics_per_epoch)

,tn,fp,fn,tp,average_precision_score,roc_auc_score,precision_score,recall_score,f1_score,accuracy_score
0,2,0,8,0,0.800,0.6875,0.0,0.000,0.000000,0.2
1,2,0,8,0,0.800,0.6250,0.0,0.000,0.000000,0.2
2,2,0,8,0,0.800,0.6250,0.0,0.000,0.000000,0.2
3,2,0,8,0,0.800,0.6250,0.0,0.000,0.000000,0.2
4,2,0,8,0,0.800,0.5625,0.0,0.000,0.000000,0.2
5,2,0,8,0,0.800,0.5000,0.0,0.000,0.000000,0.2
6,2,0,8,0,0.800,0.5000,0.0,0.000,0.000000,0.2
7,2,0,8,0,0.800,0.5000,0.0,0.000,0.000000,0.2
8,2,0,8,0,0.800,0.5000,0.0,0.000,0.000000,0.2
9,2,0,8,0,0.800,0.5000,0.0,0.000,0.000000,0.2


## Eric data

In [4]:
from google.cloud import bigquery
client = bigquery.Client()

random.seed(35)
np.random.seed(35)

In [5]:
# Get features
sql = """
SELECT
    f.* EXCEPT (asdb_plan_key, post_mnths, first_prv_dt, last_prv_dt, index_dt)
    , e.* EXCEPT (individual_id)
FROM 
    `anbc-hcb-dev.cm_medicaid_hcb_dev.a534354_IP_2024_non_embedding_features` AS f
LEFT JOIN
    `anbc-hcb-dev.cm_medicaid_hcb_dev.a534354_IP_2024_embeddings` AS e
        ON f.asdb_member_key = e.individual_id
WHERE 1=1
    AND NOT asdb_plan_key IN (33, 54)
    AND post_mnths >= 6
    
"""
df_og = client.query(sql).to_dataframe() 

df_og.shape
#2,542,308 members who qualify with 564 features we are exploring

(2542308, 561)

In [6]:
# Get labels (acute IP) stay (0=no, 1=yes) in the 6 months post-index date
sql = """
SELECT
    asdb_member_key
    , acute_ip_flag
FROM 
    `anbc-hcb-dev.cm_medicaid_hcb_dev.a534354_IP_2024_outcome_ip` AS o
WHERE 1=1 
  AND o.asdb_member_key IN (SELECT asdb_member_key FROM `anbc-hcb-dev.cm_medicaid_hcb_dev.a534354_IP_2024_non_embedding_features` WHERE 1=1 AND NOT asdb_plan_key IN (33, 54) AND post_mnths >= 6)
  
"""
outcome_og = client.query(sql).to_dataframe() 
outcome_og.shape

(2542308, 2)

In [9]:
from pandas.api.types import is_integer_dtype as is_integer
from pandas.api.types import is_float_dtype as is_float
import re

emb_pattern = r'emb[0-255]+'
emb_col = [col for col in df_og.columns if re.match(emb_pattern ,col)]
df_og[emb_col] = df_og[emb_col].fillna(0)

for c in df_og.columns: 
    dt = df_og[c].dtype
    if is_integer(dt) or is_float(dt):
        df_og[c]=df_og[c].fillna(0) 
        # print("Floatint:", dt)
    else:
        try:
            df_og[c]= df_og[c].fillna('')
        except:
            print("ERROR - DATE VARIABLE FOUND", dt)

In [8]:
#  0 := categorical, 1 := continuous, 2 := binary
nem_to_type = {
    'narc':2, 
    # 'otc_fills_yr2': 1,
    # 'otc_fills_yr1':1,  
    'COP':2, 
    'sleep_apnea':2, 
    'spinal_inj':2,
    'back':2,
    'substance':2,
    'ALC':2,
    'bipolar':2,
    'psychoses':2,
    'EDO':2, 
    'SCA':2, 
    'DIA':2, 
    'DEP':2, 
    'abdominal_pain':2, 
    'OST':2, 
    'AID':2, 
    'IDA':2, 
    'ANX':2, 
    'DEM':2, 
    'CYS':2, 
    'autoimmune':2, 
    'MOH':2, 
    'HEM':2, 
    'esrd':2, 
    'HepC':2, 
    'HYP':2, 
    'HYC':2, 
    'immune':2, 
    'intel_dsblty':2, 
    'meta_cancer':2, 
    'liver_dis':2, 
    'MSS':2, 
    'OBE':2, 
    'oud':2, 
    'liver_other':2, 
    'paralysis':2, 
    'PAR':2, 
    'PUD':0, 
    'hmd':2, 
    'PVD':2, 
    'CRO':2, 
    'AST':2, 
    'EPL':2, 
    'low_med_sev_ed_flag_yr2':2, 
    'med_high_sev_ed_flag_yr2':2, 
    'high_sev_ed_flag_yr2':2, 
    'acute_ip_flag_yr1':2, 
    'CHO':2,
    'burns':2, 
    'acute_ip_flag_yr2':2, 
    'cad':2, 
    'Cancer':2, 
    'ed_flag_yr2':2, 
    'high_sev_ed_flag_yr1':2, 
    'med_sev_ed_flag_yr2':2, 
    'AUT':2, 
    'med_high_sev_ed_flag_yr1':2, 
    'low_med_sev_ed_flag_yr1':2, 
    'low_sev_ed_flag_yr1':2, 
    'CBD':2, 
    'CHF':2, 
    'CRF':2, 
    'VNA':2, 
    'CHD':2, 
    'ed_flag_yr1':2, 
    'med_sev_ed_flag_yr1':2, 
    'low_sev_ed_flag_yr2':2, 
    'urbsubr':0, 
    'gender':0, # TODO: IF = M, 1 ELIF = F 0 ELSE NULL 
    'cms_prost_cancer_scrn':0, 
    'cms_hpv_scrn':0, 
    'cms_cvd_scrn':0, 
    'cms_lung_cancer_scrn':0, 
    'cms_pelvic':0,
    'coa_population_group':0, 
    'cms_pap':0, 
    'cms_t2d_scrn':0, 
    'cms_bone_scrn':0, 
    'cms_alc_scrn':0,
    'cms_ibt_cvd':0, 
    'cms_col_scrn':0,
    'index_dt':0, 
    'cms_ibt_obese':0, 
    'cms_flu_vax':0, 
    'cms_pneum_vax':0, 
    'cms_dep_scrn':0,
    'tenure_yr2':1, # either 
    'tenure_yr1':1, # either
    'cms_hepb_vax':0,
    'low_sev_ed_visits_yr2':0, # 7/2/24 removed
    'coa_population_category':0, 
    'cms_mam_scrn':0, # 7/2/24 removed
    'low_sev_ed_visits_yr1':1,
    'sum_acute_ip_admits_yr2':1,
    'sum_acute_ip_admits_yr1':1,
    'cms_tobacco':0,# 7/2/24 removed
    'low_med_sev_ed_visits_yr2':1, 
    'cms_t2d_train':0, # 7/2/24 removed
    'sum_preventable_yr2':1, 
    'cms_hepb_scrn':0, ###
    'sum_preventable_yr1':1, 
    'major_chronic_cnt':1, 
    'low_med_sev_ed_visits_yr1':1,
    'sum_ob':1, 
    'sum_unnecessary_yr2':1, 
    'sum_chol_lab':1, 
    'cms_nutrition':0,  # 7/2/24 removed
    'coe_anesth_clm_yr2':1, 
    'sum_a1c_lab':1, 
    'sum_unnecessary_yr1':1, 
    'sum_avoidable_yr2':1, 
    'gpi2_cnt_yr1':1, 
    'high_sev_ed_visits_yr2':1,
    'gpi2_cnt_yr2':1, 
    'ms_brand_fills_yr2':1, 
    'coe_anesth_clm_yr1':1, 
    'med_sev_ed_visits_yr2':1, 
    'cms_sti_scrn':0, 
    'med_high_sev_ed_visits_yr2':1, 
    'high_sev_ed_visits_yr1':1, 
    'ms_brand_fills_yr1':1, 
    'sum_avoidable_yr1':1, 
    'inhaled_steroid_scripts_yr2':1, 
    'med_high_sev_ed_visits_yr1':1, 
    'med_sev_ed_visits_yr1':1, 
    'inhaled_steroid_scripts_yr1':1, 
    'obs_clm_yr2':1,
    'gpi4_cnt_yr2':1,
    'gpi4_cnt_yr1':1,
    'obs_clm_yr1':1, 
    'sum_dme':1, 
    'antianginal_agent_scripts_yr2':1, 
    'mail_order_fills_yr2':1, 
    'branded_generic_fills_yr2':1, 
    'gpi_cnt_yr1':1,
    'gpi_cnt_yr2':1,
    'adi_score':1, 
    'sdi_score':1, 
    'antianginal_agent_scripts_yr1':1, 
    # 'ethnicity_code', 
    'sum_ed_visits_yr2':1,
    'mail_order_fills_yr1':1,
    'uc_clm_yr2':1, 
    'calcium_channel_blk_scripts_yr2':1,
    'antianxiety_scripts_yr2':1, 
    'beta_blocker_scripts_yr2':1, 
    'sum_ed_visits_yr1':1, 
    'branded_generic_fills_yr1':1,
    'coe_maternity_clm_yr2':1,
    'sum_chemo':1, 
    'agenbr':1, 
    'uc_clm_yr1':1,
    'calcium_channel_blk_scripts_yr1':1,
    'coe_maternity_clm_yr1':1,
    'diuretic_scripts_yr2':1,
    'primarylanguage_desc':0, # TODO: Check counts, limit to other if <2000 ppl
    'beta_blocker_scripts_yr1':1, 
    'antianxiety_scripts_yr1':1,
    'antihypertensive_scripts_yr2':1,
    'ndc_cnt_yr1':1, 
    'ndc_cnt_yr2':1, 
    'lipid_lowering_scripts_yr2':1,
    'diuretic_scripts_yr1':1, 
    'coe_surg_clm_yr2':1, 
    'sum_acute_calc_los_yr1':1, 
    'sum_acute_calc_los_yr2':1,
    'sum_pcp':1, 
    'lipid_lowering_scripts_yr1':1, 
    'antihypertensive_scripts_yr1':1,
    'coe_surg_clm_yr1':1,
    'coe_mrx_clm_yr2':1,
    'antidepressant_scripts_yr2':1,
    'coe_mrx_clm_yr1':1, 
    'antipsychotic_scripts_yr2':1, 
    'anticonvulsant_scripts_yr2':1, 
    'antidiabetic_scripts_yr2':1, 
    'antidepressant_scripts_yr1':1, 
    'coe_radio_clm_yr2':1,
    'anticonvulsant_scripts_yr1':1, 
    'coe_radio_clm_yr1':1, 
    'emis_mrx_clm_yr2':1, 
    'coe_ltc_community_clm_yr1':1, 
    'emis_community_clm_yr1':1,
    'coe_ltc_community_clm_yr2':1, 
    'emis_community_clm_yr2':1, 
    'emis_radio_clm_yr1':1, 
    'emis_radio_clm_yr2':1, 
    'antipsychotic_scripts_yr1':1, 
    'antidiabetic_scripts_yr1':1, 
    'emis_mrx_clm_yr1':1,
    'coe_ip_hos_clm_yr2':1,
    'coe_ip_hos_clm_yr1':1, 
    'coe_ip_non_hos_clm_yr2':1,
    'emis_pcp_clm_yr1':1,
    'emis_pcp_clm_yr2':1,
    'coe_phy_clm_yr1':1, 
    'coe_ip_non_hos_clm_yr1':1,
    'inhaled_steroid_days_supply_yr2':1, 
    'coe_phy_clm_yr2':1, 
    'emis_ip_clm_yr2':1,
    'ss_brand_fills_yr2':1, 
    'emis_ip_clm_yr1':1, 
    'inhaled_steroid_days_supply_yr1':1, 
    'coe_eval_clm_yr2':1, 
    'coe_ltc_ins_clm_yr2':1, 
    'emis_ins_clm_yr2':1, 
    'sum_spec':1, 
    'coe_eval_clm_yr1':1, 
    'ss_brand_fills_yr1':1,
    'retail_fills_yr2':1, 
    'retail_fills_yr1':1, 
    'emis_ins_clm_yr1':1, 
    'coe_ltc_ins_clm_yr1':1, 
    'emis_spec_clm_yr2':1,
    'emis_spec_clm_yr1':1,
    'emis_ed_clm_yr2':1,
    'coe_lab_clm_yr2':1, 
    'generic_fills_yr2':1,
    'emis_ed_clm_yr1':1,
    'last_prv_dt':1, 
    'first_prv_dt':1,
    'coe_lab_clm_yr1':1,
    'maint_drug_fills_yr2':1,
    'emis_lab_clm_yr2':1,
    'formulary_fills_yr2':1,
    'emis_lab_clm_yr1':1, 
    'generic_fills_yr1':1, 
    'coe_op_hos_clm_yr2':1, 
    'formulary_fills_yr1':1,
    'emis_misc_clm_yr2':1,
    'maint_drug_fills_yr1':1,
    'coe_op_hos_clm_yr1':1, 
    'rx_claim_cnt_yr2':1, 
    'antianginal_agent_days_supply_yr2':1,
    'coe_mh_clm_yr2':1,
    'coe_mh_clm_yr1':1,
    'emis_misc_clm_yr1':1, 
    'coe_ltc_home_clm_yr2':1,
    'emis_home_clm_yr2':1,
    'antianginal_agent_days_supply_yr1':1, 
    'ltc_clm_yr2':1,
    'coe_ltc_home_clm_yr1':1,
    'emis_home_clm_yr1':1, 
    'emis_hh_clm_yr2':1, 
    'emis_hh_clm_yr1':1, 
    'ltc_clm_yr1':1, 
    'rx_claim_cnt_yr1':1, 
    'calcium_channel_blk_days_supply_yr2':1, 
    'sum_op_visits_yr2':1, 
    'sum_op_visits_yr1':1, 
    'coe_op_non_hos_clm_yr2':1,
    'emis_ambul_clm_yr2':1, 
    'water_quality':1, 
    'coe_op_non_hos_clm_yr1':1, 
    'calcium_channel_blk_days_supply_yr1':1, 
    'beta_blocker_days_supply_yr2':1,
    'emis_ambul_clm_yr1':1,
    'beta_blocker_days_supply_yr1':1, 
    'emis_mh_clm_yr2':1, 
    'emis_mh_clm_yr1':1, 
    'diuretic_days_supply_yr2':1, 
    'antianxiety_days_supply_yr2':1, 
    'lipid_lowering_days_supply_yr2':1, 
    'coe_other_clm_yr2':1, 
    'coe_other_clm_yr1':1,
    'antianxiety_days_supply_yr1':1, 
    'antihypertensive_days_supply_yr2':1,
    'diuretic_days_supply_yr1':1, 
    'lipid_lowering_days_supply_yr1':1, 
    'antihypertensive_days_supply_yr1':1, 
    'antipsychotic_days_supply_yr2':1,
    'antidepressant_days_supply_yr2':1,
    'antipsychotic_days_supply_yr1':1,
    'antidepressant_days_supply_yr1':1, 
    'anticonvulsant_days_supply_yr2':1, 
    'anticonvulsant_days_supply_yr1':1,
    'antidiabetic_days_supply_yr2':1, 
    'antidiabetic_days_supply_yr1':1, 
    'income_inequality':1, 
    'svi_score':1, 
    'days_supply_sum_yr2':1, 
    'zip_weight_avg_medinc':1,
    'days_supply_sum_yr1':1,
    'food_access':1, 
    'citizenship_index':1,
    'acs_social_risk_score':1, 
    'housing_desert':1, 
    'unemployment_index':1,
    'health_habits':1,
    'natural_disaster':1,
    'proactive_health':1,
    'housing_ownership':1, 
    'health_infra':1,
    'language_score':1, 
    'racial_diversity':1, 
    'housing_quality':1,
    'income_index':1,
    'education_index':1,
    'transport_access':1, 
    'disability_score':1,
    'technology_access':1,
    'poverty_score':1, 
    'social_isolation':1,
    'health_access':1,
    'csdi_social_risk_score':1,
     'asdb_member_key':1,
}

In [14]:
index_to_feature = dict(enumerate(df_og.columns))
feature_to_index = {value: key for key, value in index_to_feature.items()}
categorical_features = [feature for feature in nem_to_type if nem_to_type[feature] == 0]
categorical_indices = [feature_to_index[feature] for feature in categorical_features if feature in feature_to_index]
len(categorical_features)

NameError: name 'df_og' is not defined

In [13]:
# ONE HOT ENCODING
categorical_features.remove('index_dt')
df_og[categorical_features] = df_og[categorical_features].astype(str)
df_og['gender'] = df_og['gender'].map({'M': 1, 'F': 0}).fillna(-1)  
categorical_features.remove('gender')

In [14]:
# ONE HOT ENCODING
from tqdm.notebook import tqdm
from sklearn.preprocessing import OneHotEncoder
min_occurrence = 2000

encoder = OneHotEncoder(sparse_output=False)

for feature in tqdm(categorical_features):
    counts = df_og[feature].value_counts()
    categories_to_keep = counts[counts >= min_occurrence].index
   
    filtered_df = df_og[df_og[feature].isin(categories_to_keep)]
   
    if not filtered_df.empty:
        encoded_data = encoder.fit_transform(filtered_df[[feature]])
        encoded_df = pd.DataFrame(encoded_data, columns=encoder.get_feature_names_out([feature]))
       
        df_og = df_og.drop(feature, axis=1)
        df_og = pd.concat([df_og, encoded_df], axis=1)

  0%|          | 0/28 [00:00<?, ?it/s]

In [15]:
df_og.set_index('asdb_member_key', inplace = True)
outcome_og.set_index('asdb_member_key', inplace = True)
merged = df_og.merge(outcome_og, on='asdb_member_key', how='left')

In [18]:
merged.to_feather("cacm_mdcd_new_member_feature_outcoem_4_modeling_data.feather")

### Read in data

In [5]:
merged = pd.read_feather("cacm_mdcd_new_member_feature_outcoem_4_modeling_data.feather")

In [6]:
string_columns = merged.select_dtypes(include='object').columns.tolist()

In [7]:
from sklearn.preprocessing import LabelEncoder
label_encoder = LabelEncoder()
for col in string_columns:
    merged[col] = label_encoder.fit_transform(merged[col])

merged = merged.astype(float)

In [8]:
merged.fillna(0, inplace = True)

In [9]:
features = merged.drop("acute_ip_flag", axis = 1).values

In [10]:
outcomes = merged['acute_ip_flag'].astype(float).values

In [11]:
# All available supported metrics
eval_metrics = ['confusion_matrix', 
                 'average_precision_score', 
                 'roc_auc_score', 
                'precision_score', 
                 'recall_score', 
                 'f1_score', 
                 'lift',
                 'accuracy_score']

#### Feature selection

In [11]:
import gc
from xgboost import XGBClassifier
from sklearn.feature_selection import RFECV
from sklearn.model_selection import StratifiedKFold
# Check if GPU is available
import tensorflow as tf
gpus = tf.config.experimental.list_physical_devices('GPU')
if not gpus:
    raise SystemError("No GPUs found. Please ensure your environment has a GPU available.")

2024-08-05 04:41:24.149599: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:485] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
2024-08-05 04:41:24.864409: E external/local_xla/xla/stream_executor/cuda/cuda_dnn.cc:8454] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
2024-08-05 04:41:25.190097: E external/local_xla/xla/stream_executor/cuda/cuda_blas.cc:1452] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
2024-08-05 04:41:27.327601: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
I0000 00:00:1722832910.403231    3890 cuda_executor.c

In [ ]:
np_xg = merged.drop("acute_ip_flag", axis = 1).values

In [24]:
X_train, X_test, y_train, y_test = train_test_split(np_xg, outcomes, test_size=0.2, random_state=35, stratify=outcomes)
X_test, X_val, y_test, y_val = train_test_split(X_test, y_test, test_size=0.5, random_state=35, stratify=y_test)

In [65]:
!pip install tensorflow

  Using cached tensorboard-2.17.0-py3-none-any.whl.metadata (1.6 kB)
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 47.8/47.8 kB 3.6 MB/s eta 0:00:00
  Using cached Markdown-3.6-py3-none-any.whl.metadata (7.0 kB)
  Using cached tensorboard_data_server-0.7.2-py3-none-manylinux_2_31_x86_64.whl.metadata (1.1 kB)
  Using cached werkzeug-3.0.3-py3-none-any.whl.metadata (3.7 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 601.3/601.3 MB 1.5 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 57.5/57.5 kB 7.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.3/5.3 MB 115.4 MB/s eta 0:00:0000:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.1/1.1 MB 66.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.5/24.5 MB 17.0 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.2/2.2 MB 4.6 MB/s eta 0:00:000:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 65.5/65.5 kB 363.6 kB/s eta 0:00:00a 0:00:01
Using cached tensorboard-2.17.0-py

In [ ]:
# Initialize the GPU-enabled XGBoost classifier
xgb_model_rfe = XGBClassifier(
    random_state=53,
    n_jobs=-1,
    verbosity=0,
    use_label_encoder=False,
    tree_method = "hist", 
    device = "cuda"
)

# Create RFECV with GPU-enabled XGBoost
rfecv = RFECV(
    estimator=xgb_model_rfe,
    step=10,
    cv=StratifiedKFold(3),
    scoring="roc_auc",
    min_features_to_select=1,
    n_jobs=-1
)

# Fit RFECV and manage memory
rfecv.fit(X_train, y_train)

# Print the optimal number of features
print(f"Optimal number of features: {rfecv.n_features_}")

# Transform datasets to include only the selected features
# X_train_selected = rfecv.transform(X_train)
# X_val_selected = rfecv.transform(X_val)
# X_test_selected = rfecv.transform(X_test)

# Clear memory after fitting
gc.collect()

In [ ]:
n_features = list(range(1, len(rfecv.cv_results_['mean_test_score']) + 1))
mean_test_scores = rfecv.cv_results_['mean_test_score']
cv_results = pd.DataFrame({
    'n_features': n_features,
    'mean_test_score': mean_test_scores
})

plt.figure()
plt.xlabel("Number of features selected")
plt.ylabel("Mean test accuracy")
plt.plot(cv_results["n_features"], cv_results["mean_test_score"])
plt.title("Recursive Feature Elimination \nwith correlated features")
plt.show()


#### Get selected features for Xgboost

In [14]:
f = open('xgboost_selected_features.txt', 'r')
xgboost_selected_features = selected_features = [x.strip() for x in f.readlines()]
print(len(xgboost_selected_features))
f.close()

59


In [15]:
np_featurs_selected_xg =merged[xgboost_selected_features].values

In [16]:
X_train, X_test, y_train, y_test = train_test_split(np_featurs_selected_xg, outcomes, test_size=0.2, random_state=35, stratify=outcomes)
X_test, X_val, y_test, y_val = train_test_split(X_test, y_test, test_size=0.5, random_state=35, stratify=y_test)

In [17]:
X_test.shape

(254231, 59)

In [18]:
y_test.shape

(254231,)

In [19]:
from imblearn.under_sampling import RandomUnderSampler
rus_sampler = RandomUnderSampler(random_state = 44, sampling_strategy = 0.03)
X_train_rus, y_train_rus = rus_sampler.fit_resample(X_train, y_train)

In [20]:
import logging
import optuna
import xgboost as xgb
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import train_test_split

def objective(trial):
    params = {
        'learning_rate': trial.suggest_float('learning_rate', 0.001, 0.3, log=True),
        'n_estimators': trial.suggest_int('n_estimators', 100, 7000),
        'max_depth': trial.suggest_int('max_depth', 4, 12),
        'subsample': trial.suggest_float('subsample', 0.5, 1.0),
        'colsample_bytree': trial.suggest_float('colsample_bytree', 0.5, 1.0),
        'random_state': 53, 
        'tree_method' : "hist", 
        'device' : "cuda",
        'n_jobs': -1
    }
    
    model = xgb.XGBClassifier(**params)
    model.fit(X_train_rus, y_train_rus, eval_set=[(X_val, y_val)], verbose=0)
   
    y_pred_test = model.predict_proba(X_test)[:, 1]
    roc_auc_test = roc_auc_score(y_test, y_pred_test)
    eval_metrics_res = evaluate_scores(y_test.reshape(-1), y_pred_test, eval_metrics)
   
    print(f"Trial {trial.number} AUC: {roc_auc_test}; eva_metrics {eval_metrics_res}, Params: {params}")
   
    # study_results = study.trials_dataframe(attrs=('number', 'value', 'params', 'state'))  # Saving the results to a CSV file
    # study_results.to_csv('optuna_results.csv', index=False)
   
    return roc_auc_test

study = optuna.create_study(direction="maximize")
study.optimize(objective, n_trials=75)  # trials must be at least 50
print("Number of finished trials: ", len(study.trials))

best_trial = study.best_trial
print("Best trial:")
print(" AUC:", best_trial.value)
print(" Params:", best_trial.params)

# Log the best result
print(f"Best AUC: {best_trial.value} with params: {best_trial.params}")
study_results = study.trials_dataframe(attrs=('number', 'value', 'params', 'state'))  # Saving the results to a CSV file

[I 2024-08-05 04:48:20,739] A new study created in memory with name: no-name-86b473f1-8a3d-451c-8c7c-6805d4de2186
/opt/conda/envs/pytorch/lib/python3.10/site-packages/xgboost/core.py:160: UserWarning: [04:49:45] WARNING: /workspace/src/common/error_msg.cc:58: Falling back to prediction using DMatrix due to mismatched devices. This might lead to higher memory usage and slower performance. XGBoost is running on: cuda:0, while the input data is on: cpu.
Potential solutions:
- Use a data structure that matches the device ordinal in the booster.
- Set the device for booster before call to inplace_predict.

This warning will only be shown once.

  warnings.warn(smsg, UserWarning)
[I 2024-08-05 04:49:47,696] Trial 0 finished with value: 0.8584893001348831 and parameters: {'learning_rate': 0.04529328473133285, 'n_estimators': 3639, 'max_depth': 7, 'subsample': 0.8367389385065691, 'colsample_bytree': 0.8184074348936714}. Best is trial 0 with value: 0.8584893001348831.


Trial 0 AUC: 0.8584893001348831; eva_metrics {'tn': 126747, 'fp': 123125, 'fn': 369, 'tp': 3990, 'average_precision_score': 0.03018318865460859, 'roc_auc_score': 0.8584893001348831, 'precision_score': 0.03138889981512803, 'recall_score': 0.9153475567790778, 'f1_score': 0.06069641145777872, 'lift': 1.8307023144986958, 'accuracy_score': 0.5142449189909963}, Params: {'learning_rate': 0.04529328473133285, 'n_estimators': 3639, 'max_depth': 7, 'subsample': 0.8367389385065691, 'colsample_bytree': 0.8184074348936714, 'random_state': 53, 'tree_method': 'hist', 'device': 'cuda', 'n_jobs': -1}


[I 2024-08-05 04:54:26,460] Trial 1 finished with value: 0.8694137826647078 and parameters: {'learning_rate': 0.0012662620033393195, 'n_estimators': 4942, 'max_depth': 11, 'subsample': 0.6568616071462261, 'colsample_bytree': 0.6230796918063746}. Best is trial 1 with value: 0.8694137826647078.


Trial 1 AUC: 0.8694137826647078; eva_metrics {'tn': 126815, 'fp': 123057, 'fn': 301, 'tp': 4058, 'average_precision_score': 0.030903388436055564, 'roc_auc_score': 0.8694137826647078, 'precision_score': 0.03192384848365653, 'recall_score': 0.9309474650149117, 'f1_score': 0.06173083651520453, 'lift': 1.8619022536931595, 'accuracy_score': 0.5147798655553414}, Params: {'learning_rate': 0.0012662620033393195, 'n_estimators': 4942, 'max_depth': 11, 'subsample': 0.6568616071462261, 'colsample_bytree': 0.6230796918063746, 'random_state': 53, 'tree_method': 'hist', 'device': 'cuda', 'n_jobs': -1}


[W 2024-08-05 04:54:41,951] Trial 2 failed with parameters: {'learning_rate': 0.013235967035484972, 'n_estimators': 4641, 'max_depth': 5, 'subsample': 0.7817604066902986, 'colsample_bytree': 0.7733260050965269} because of the following error: KeyboardInterrupt().
Traceback (most recent call last):
  File "/opt/conda/envs/pytorch/lib/python3.10/site-packages/optuna/study/_optimize.py", line 196, in _run_trial
    value_or_values = func(trial)
  File "/var/tmp/ipykernel_3890/264419064.py", line 21, in objective
    model.fit(X_train_rus, y_train_rus, eval_set=[(X_val, y_val)], verbose=0)
  File "/opt/conda/envs/pytorch/lib/python3.10/site-packages/xgboost/core.py", line 730, in inner_f
    return func(**kwargs)
  File "/opt/conda/envs/pytorch/lib/python3.10/site-packages/xgboost/sklearn.py", line 1519, in fit
    self._Booster = train(
  File "/opt/conda/envs/pytorch/lib/python3.10/site-packages/xgboost/core.py", line 730, in inner_f
    return func(**kwargs)
  File "/opt/conda/envs/pyto

KeyboardInterrupt: 

In [39]:
f = open('catboost_selected_features.txt', 'r')
catboost_selected_features = selected_features = [x.strip() for x in f.readlines()]
print(len(catboost_selected_features))
f.close()

499


In [40]:
np_sampled_cat=df_sampled.astype('float32').to_numpy()
np_sampled_cat = np.where(np.isnan(np_sampled_cat), 0, np_sampled_cat)

In [42]:
np_sampled_xg=df_sampled[xgboost_selected_features].astype('float32').to_numpy()
np_sampled_xg = np.where(np.isnan(np_sampled_xg), 0, np_sampled_xg)

### Pilot

In [12]:
from sklearn.metrics import roc_auc_score

In [13]:
# All available supported metrics
eval_metrics = ['confusion_matrix', 
                 'average_precision_score', 
                 'roc_auc_score', 
                'precision_score', 
                 'recall_score', 
                 'f1_score', 
                 'lift',
                 'accuracy_score']

In [14]:
X_train, X_test, y_train, y_test = train_test_split(features, outcomes, test_size=0.2, random_state=35, stratify=outcomes)
X_test, X_val, y_test, y_val = train_test_split(X_test, y_test, test_size=0.5, random_state=35, stratify=y_test)

In [21]:
train_dataset = EmbeddingDatasets(X_train.astype('float32'), y_train)
val_dataset = EmbeddingDatasets(X_val.astype('float32'), y_val)
test_dataset = EmbeddingDatasets(X_test.astype('float32'), y_test)

In [22]:
train_dataloader = DataLoader(train_dataset, batch_size=32, shuffle=True)
val_dataloader = DataLoader(val_dataset, batch_size=32, shuffle=False)
test_dataloader = DataLoader(test_dataset, batch_size=32, shuffle=False)

#### Simple model

In [23]:
n_hidden1 = 256  # Number of hidden nodes
n_hidden2 = 128
n_output =  1   # Number of output nodes = for binary classifier
class MLPModel(torch.nn.Module):
    
    def __init__(self, input_dim):
        super(MLPModel, self).__init__()
        self.layer_1 = nn.Linear(input_dim, n_hidden1) 
        self.batchnorm1 = nn.BatchNorm1d(n_hidden1)
        self.relu1 = nn.ReLU()
        
        self.layer_2 = nn.Linear(n_hidden1, n_hidden2)
        self.batchnorm2 = nn.BatchNorm1d(n_hidden2)
        self.relu2 = nn.ReLU()
        self.dropout = nn.Dropout(p=0.2)

        self.layer_out = nn.Linear(n_hidden2, 1) 
        self.sigmoid =  nn.Sigmoid()
    
    def forward(self, inputs):
        
        x = self.layer_1(inputs)
        x = self.batchnorm1(x)
        x = self.relu1(x)
        x = self.layer_2(x)
        x = self.batchnorm2(x)
        x = self.relu2(x)
        x = self.dropout(x)
        x = self.layer_out(x)
        out = self.sigmoid(x)
        
        return out

#### Training and testing

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
device

In [24]:
def model_train(model, X_train, y_train, X_val, y_val, epochs):
    
    train_dataset = EmbeddingDatasets(X_train.astype('float32'), y_train)
    val_dataset = EmbeddingDatasets(X_val.astype('float32'), y_val)
    train_dataloader = DataLoader(train_dataset, batch_size=32, shuffle=True)
    val_dataloader = DataLoader(val_dataset, batch_size=32, shuffle=False)    
    train_loss_history = []
    val_loss_history = []
    val_metrics_history = []
    for epoch in tqdm(range(epochs)):
        #Within each epoch run the subsets of data = batch sizes.
        model.train(True)
        train_loss = 0
        for batch in train_dataloader:
            inputs = batch['embedding'].to(device)
            labels = batch['label'].float().unsqueeze(1).to(device)
            y_pred = model(inputs)            # Forward Propagation
            loss = loss_func(y_pred, labels)  # Loss Computation
            optimizer.zero_grad()         # Clearing all previous gradients, setting to zero 
            loss.backward()               # Back Propagation
            optimizer.step()              # Updating the parameters 
            train_loss += loss.item()
        train_loss_history.append(train_loss/len(train_dataloader))
        
        val_loss = 0
        val_roc_auc = 0
        val_lift1perc = 0
        model.eval()
        with torch.no_grad():
            for batch in val_dataloader:
                inputs = batch['embedding'].to(device)
                labels = batch['label'].float().unsqueeze(1).to(device)
                y_pred = model(inputs)
                loss = loss_func(y_pred, labels)
                val_loss += loss.item()
                y_pred = y_pred.detach().cpu().numpy().reshape(-1)
                labels = labels.detach().cpu().numpy().reshape(-1)
                val_lift1perc += lift_at_x_percent(labels, 
                                                  y_pred)
                try: 
                    val_roc_auc += roc_auc_score(labels, 
                                                 y_pred)
                except ValueError:
                    val_roc_auc += 0.5
                    pass
        val_loss_history.append(val_loss/len(val_dataloader))
        val_metrics_history.append([val_lift1perc/len(val_dataloader), val_roc_auc/len(val_dataloader)])
            
        scheduler.step(val_loss/len(val_dataloader))

    return model, train_loss_history, val_loss_history, val_metrics_history


In [25]:
#Loss Computation
loss_func = nn.BCELoss()
#Optimizer
learning_rate = 0.01
epochs = 20
eval_metrics = ['confusion_matrix', 
                 'average_precision_score', 
                 'roc_auc_score', 
                'precision_score', 
                 'recall_score', 
                 'f1_score', 
                 'lift',
                 'accuracy_score']

In [26]:
from time import time
from datetime import datetime
datetime

datetime.datetime

In [ ]:
ratio_result = {}
test_ratio_models = {}

In [ ]:
# Has not tested
for rus_ratio in tqdm([
    # 0.03, 0.05, 0.1, 0.3, 
                       0.5, 1, 10]): 
    sampler = RandomUnderSampler(random_state = 44, sampling_strategy = rus_ratio)
    X_train_rus, y_train_rus = sampler.fit_resample(X_train, y_train)
    input_dim = features.shape[1]
    model = MLPModel(input_dim = input_dim).to(device)
    model, train_loss_history, val_loss_history, val_metrics_history = model_train(model, 
                                                                                   X_train_rus, y_train_rus, 
                                                                                   X_val, y_val, 
                                                                                   epochs = 20)
    print(val_metrics_history)
    y_pred_proba = model(torch.from_numpy(X_test).float().to(device))
    y_pred_proba = y_pred_proba.detach().cpu().numpy()
    ratio_result[f"{rus_ratio}"] = lift_at_x_percent(y_test.reshape(-1), y_pred_proba.reshape(-1))
    test_ratio_models[f"{rus_ratio}"] = model
    del X_train_rus
    del y_train_rus
    del model
    

100%|██████████| 20/20 [33:43<00:00, 101.19s/it]


[[4.7786868051185225, 0.5278880855271281], [4.86784140969163, 0.5294408886273658], [5.028319697923223, 0.5359824339458449], [4.314033983637507, 0.5168206866539344], [4.9653870358716174, 0.5293497218913841], [4.529053912313825, 0.5206573474817235], [4.950702748059577, 0.5333236084127094], [4.780784560520243, 0.5250221011200624], [4.608768617579189, 0.5213571421015543], [4.639186070904132, 0.5250883430424662], [4.485001048877701, 0.5205523672818027], [4.61401300608349, 0.5205338778422915], [4.596182085168869, 0.5215906476825451], [4.221732745961821, 0.5155020721357985], [4.799664359135725, 0.5315941889156729], [4.832179567862387, 0.5284002914526987], [4.501783092091461, 0.5207043930736036], [4.662261380323054, 0.5225775818928253], [4.673799035032515, 0.5222046048936758], [4.807006503041745, 0.5266361476812684]]


100%|██████████| 20/20 [23:51<00:00, 71.59s/it]


[[0.7856093979441997, 0.46503939228666286], [0.7740717432347388, 0.46397541783609864], [0.7037969372771135, 0.4633624726492582], [0.7835116425424796, 0.463573525067517], [0.7163834696874345, 0.4656191099661274], [0.7667295993287183, 0.4643844549531843], [0.7835116425424795, 0.46437957042249384], [0.7604363331235579, 0.4642321679373694], [0.8002936857562409, 0.46583466368590243], [0.6975036710719531, 0.4651431752600168], [0.7583385777218377, 0.46317773758496533], [0.7656807216278583, 0.46192649990761064], [0.777218376337319, 0.46260571771259906], [0.8044891965596812, 0.4631529181184356], [0.775120620935599, 0.46304528451923704], [0.7656807216278583, 0.46457737675266547], [0.8244178728760226, 0.4641037970443172], [0.7153345919865745, 0.4644622060476104], [0.7646318439269981, 0.46368097515895224], [0.707992448080554, 0.4611237138571693]]


100%|██████████| 20/20 [15:21<00:00, 46.07s/it]


[[4.768198028109922, 0.5317653439981249], [5.356618418292431, 0.5408552705946127], [5.206628907069438, 0.540465939125892], [5.3555695405915715, 0.5453498728922175], [5.347178518984689, 0.5443790943653218], [5.080763582966228, 0.5389254014055911], [5.434235368156077, 0.5425528428463647], [5.488777008600802, 0.5449811735925332], [5.324103209565768, 0.5452852282781202], [5.4447241451646775, 0.5463029755047546], [5.388084749318233, 0.5466255023328248], [5.1143276693937505, 0.538999168044438], [5.3576672959932905, 0.5437717605176936], [5.435284245856937, 0.5444393043196113], [5.486679253199081, 0.5464688364743765], [4.9402139710509765, 0.5404988419991674], [5.372351583805333, 0.5450767651638345], [5.507656807216282, 0.5417374848474611], [5.5160478288231625, 0.5507119903857134], [5.4268932242500565, 0.5454616785290028]]


100%|██████████| 20/20 [09:38<00:00, 28.95s/it]


[[4.3643801132787905, 0.535065117328538], [4.535347178518984, 0.5333993084111179], [4.353891336270189, 0.5293897311856299], [4.436752674638135, 0.5332028367022156], [4.712607509964338, 0.5384178930213206], [4.647577092511013, 0.5332551307105486], [4.526956156912103, 0.5343417111080837], [4.336060415355568, 0.5353430582243268], [4.514369624501783, 0.532272624863514], [4.3696245017830915, 0.5366441897899696], [4.242710299979022, 0.5273853833369008], [4.539542689322425, 0.5360715308905721], [4.627648416194671, 0.5327539359729738], [4.573106775749947, 0.5368842345179371], [4.855254877281309, 0.5374832987750205], [4.581497797356828, 0.5320338314662574], [4.562617998741347, 0.5329582276515494], [4.419970631424373, 0.5309549802289979], [4.516467379903503, 0.533349561056804], [4.654919236417033, 0.5367757464011113]]


 80%|████████  | 4/5 [1:24:02<21:00, 1260.73s/it]

KeyboardInterrupt



In [56]:
y_pred_score_rus_01 = test_ratio_models['0.1'](torch.from_numpy(X_test).float().to(device)).detach().cpu().numpy()
true_positive_at_1_percent(y_test.reshape(-1), 
                              y_pred_score_rus_01.reshape(-1))

0.0662996100022941

In [28]:
input_dim = features.shape[1]
base_model = MLPModel(input_dim = input_dim).to(device)
print(base_model)

MLPModel(
  (layer_1): Linear(in_features=624, out_features=256, bias=True)
  (batchnorm1): BatchNorm1d(256, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
  (relu1): ReLU()
  (layer_2): Linear(in_features=256, out_features=128, bias=True)
  (batchnorm2): BatchNorm1d(128, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
  (relu2): ReLU()
  (dropout): Dropout(p=0.2, inplace=False)
  (layer_out): Linear(in_features=128, out_features=1, bias=True)
  (sigmoid): Sigmoid()
)


In [29]:
optimizer = torch.optim.Adam(model.parameters(), lr=learning_rate)
scheduler = ReduceLROnPlateau(optimizer, 'min')

In [ ]:
trained_model, train_loss_history, val_loss_history, val_metrics_history = model_train(model, 
                                                                               X_train, y_train, 
                                                                               X_val, y_val, 
                                                                               epochs = epochs)

  0%|          | 0/20 [00:00<?, ?it/s]

In [32]:
val_metrics_history

[[13.655758338577707, 0.6481246173750258],
 [13.986154814348636, 0.6494760297342126],
 [13.880218166561766, 0.6495870246607796],
 [14.374239563666862, 0.6514682073088793],
 [14.314453534717838, 0.6530966882028787],
 [14.162366268093125, 0.653405587945266],
 [14.33228445563246, 0.6535581966135756],
 [13.886511432766927, 0.6522070256812676],
 [14.391021606880622, 0.6526038603840418],
 [14.178099433606027, 0.6503272846404428],
 [14.441367736521908, 0.6545194578304804],
 [14.21795678623871, 0.6540254545545118],
 [13.994545835955513, 0.6532670313250626],
 [14.604992657856076, 0.6543594900731039],
 [14.526326830291572, 0.6547083014481949],
 [14.427732326410725, 0.655079716936848],
 [14.32808894482902, 0.6538085954739652],
 [14.232641074050752, 0.6546638087420792],
 [14.505349276274368, 0.6536258712076929],
 [14.24627648416193, 0.6543133801504313]]

In [36]:
y_pred_score = trained_model(torch.from_numpy(X_test).float().to(device)).detach().cpu().numpy()
true_positive_at_x_percent(y_test.reshape(-1), 
                              y_pred_score.reshape(-1),
                             ratio = 0.01)

0.1984400091764166

In [37]:
precision_at_x_percent(y_test.reshape(-1), 
                          y_pred_score.reshape(-1),
                          ratio = 0.01)

0.34028324154209283

In [38]:
lift_at_x_percent(y_test.reshape(-1), 
                  y_pred_score.reshape(-1),
                  ratio = 0.01)

19.844000917641658

In [54]:
# Set the version number
version = 1.0

# Get the current date and time
now = datetime.now()
date_time_str = now.strftime("%Y-%m-%d_%H-%M-%S")

# Create the filename
filename = f"new_member_dnn_model_v{version}_{date_time_str}.pth"

# Save the model
torch.save(trained_model.state_dict(), filename)


In [64]:
input_dim = features.shape[1]
old_model = MLPModel(input_dim = input_dim).to(device)
old_model.load_state_dict(torch.load("new_member_dnn_model_v1.0_2024-08-05_16-21-03.pth"))
old_model.eval()

/var/tmp/ipykernel_4301/3230300997.py:3: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  old_model.load_state_dict(torch.load("new_member_dnn_model_v1.0_2024-08-05_16-21-03.pt

MLPModel(
  (layer_1): Linear(in_features=624, out_features=256, bias=True)
  (batchnorm1): BatchNorm1d(256, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
  (relu1): ReLU()
  (layer_2): Linear(in_features=256, out_features=128, bias=True)
  (batchnorm2): BatchNorm1d(128, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
  (relu2): ReLU()
  (dropout): Dropout(p=0.2, inplace=False)
  (layer_out): Linear(in_features=128, out_features=1, bias=True)
  (sigmoid): Sigmoid()
)

In [77]:
y_pred_score = old_model(torch.from_numpy(X_test).float().to(device)).detach().cpu().numpy()
true_positive_at_x_percent(y_test.reshape(-1), 
                          y_pred_score.reshape(-1),
                          ratio = 0.1)

0.6171140169763707

In [79]:
precision_at_x_percent(y_test.reshape(-1), 
                          y_pred_score.reshape(-1),
                          ratio = 0.1)

0.10580969987806317

In [78]:
lift_at_x_percent(y_test.reshape(-1), 
                  y_pred_score.reshape(-1),
                  ratio = 0.1)

6.1711401697637065

### Run pipeline (Not run because it takes really long)

In [68]:
param_space = {
    'hidden_sizes': tune.choice([[512], [256], [128], [64], 
                                 [512, 256], [256, 128], [512, 128],
                                 [256, 64], [128, 64], [64, 32]]),
    'dropout_rate': tune.uniform(0.1, 0.4),
    'batch_size': tune.choice([16, 32, 64]),
    'lr': tune.loguniform(1e-4, 1e-2),
    'optimizer': tune.choice(['adam', 'sgd']),
    'num_epochs': tune.choice([20, 35, 50, 70]),
    'momentum': tune.uniform(0.90, 0.99)
}
# All available supported metrics
eval_metrics = ['confusion_matrix', 
                'average_precision_score', 
                'roc_auc_score', 
                'precision_score', 
                'recall_score', 
                'f1_score', 
                'lift',
                'lift@1percent',
                'precision@1percent',
                'recall@1percent',
                 'accuracy_score']


In [69]:
evaluator = DNNEvaluator(embeddings=features, 
                         labels=outcomes,
                         rebalance = False,
                         val_size = 0.10,
                         test_size = 0.10
                        )
val_metrics, test_metrics = evaluator.perform_evaluation(eval_metrics=eval_metrics, 
                             param_space = param_space, 
                             num_trials=1)

2024-08-06 14:51:38,174	INFO worker.py:1744 -- Started a local Ray instance. View the dashboard at 127.0.0.1:8265 
2024-08-06 14:51:40,696	INFO tune.py:253 -- Initializing Ray automatically. For cluster usage or custom Ray initialization, call `ray.init(...)` before `Tuner(...)`.
2024-08-06 14:51:40,700	INFO tune.py:616 -- [output] This uses the legacy output and progress reporter, as Jupyter notebooks are not supported by the new engine, yet. For more information, please see https://github.com/ray-project/ray/issues/36949
Epochs:   3%|▎         | 1/35 [11:41<6:37:38, 701.73s/it]
2024-08-06 15:06:19,200	WARNING tune.py:219 -- Stop signal received (e.g. via SIGINT/Ctrl+C), ending Ray Tune run. This will try to checkpoint the experiment state one last time. Press CTRL+C (or send SIGINT/SIGKILL/SIGTERM) to skip. 
2024-08-06 15:06:19,206	INFO tune.py:1009 -- Wrote the latest version of all result files and experiment state to '/home/jupyter/ray_results/hp_train_func_2024-08-06_14-51-28' in

== Status ==
Current time: 2024-08-06 15:06:19 (running for 00:14:38.31)
Using FIFO scheduling algorithm.
Logical resource usage: 1.0/16 CPUs, 0/2 GPUs (0.0/1.0 accelerator_type:T4)
Result logdir: /var/tmp/ray/session_2024-08-06_14-51-28_794143_4301/artifacts/2024-08-06_14-51-40/hp_train_func_2024-08-06_14-51-28/driver_artifacts
Number of trials: 1/1 (1 RUNNING)
+------------------------+----------+-------------------+--------------+----------------+----------------+------------+------------+--------------+-------------+
| Trial name             | status   | loc               |   batch_size |   dropout_rate | hidden_sizes   |         lr |   momentum |   num_epochs | optimizer   |
|------------------------+----------+-------------------+--------------+----------------+----------------+------------+------------+--------------+-------------|
| hp_train_func_45564d50 | RUNNING  | 10.112.1.28:86128 |           16 |       0.179899 | (256, 64)      | 0.00138693 |   0.957863 |           35 | a

KeyboardInterrupt: 

In [56]:
test_metrics

{'tn': 27508,
 'fp': 27446,
 'fn': 270,
 'tp': 332,
 'average_precision_score': 0.011451376827523587,
 'roc_auc_score': 0.5346610792693183,
 'precision_score': 0.011951904384764922,
 'recall_score': 0.5514950166112956,
 'f1_score': 0.02339675828047921,
 'lift': 1.1029900332225915,
 'accuracy_score': 0.5011159910720714}

In [70]:
test_metrics

{'tn': 27509,
 'fp': 27445,
 'fn': 269,
 'tp': 333,
 'average_precision_score': 0.01147314409524591,
 'roc_auc_score': 0.5414214147332164,
 'precision_score': 0.011987904096767225,
 'recall_score': 0.553156146179402,
 'f1_score': 0.02346723044397463,
 'lift': 1.1063122923588038,
 'accuracy_score': 0.5011519907840737}